# RM2 Temporal Activity Profile

Notebook ini menambahkan analisis temporal untuk RM2: weekday activity, jam lokal, daypart, account-level profile, HCC-level profile, brand-context profile, video-normalized metrics, recurrence mingguan, dan uji distribusi. Notebook ini hanya membaca output yang sudah tersedia dan tidak menjalankan ulang RM1, LCN, Louvain, FSA_V, model sentimen, WordCloud, atau aturan actor type.

## 1. Judul dan Tujuan

Pertanyaan utama: pada hari dan jam apa Individual Actor, Community Actor, Mass Actor, akun anggota HCC, dan HCC paling aktif berkomentar; bagaimana pola itu berbeda menurut konteks brand; dan apakah pola tersebut muncul berulang pada beberapa minggu.

## 2. Posisi Analisis Temporal Dalam RM2

Analisis temporal ini adalah tambahan deskriptif setelah RM2 sentiment-goals dan RM2 actor typology. Input aktor, sentimen, HCC, dan konteks brand diambil dari output yang sudah ada.

## 3. Batasan Interpretasi

Analisis hari dan jam aktivitas hanya menggambarkan distribusi waktu komentar yang teramati. Konsentrasi aktivitas pada hari atau jam tertentu tidak membuktikan adanya jadwal kerja, hubungan komersial, koordinasi terencana, maupun status akun sebagai buzzer.

Istilah analitis yang digunakan: Community Actor, akun anggota HCC, aktivitas komentar HCC, pola temporal yang teramati, konsentrasi aktivitas berdasarkan hari, weekday activity, dan observed temporal pattern. Istilah buzzer hanya muncul dalam catatan keterbatasan bahwa penelitian belum membuktikan suatu akun sebagai buzzer.

## 4. Import dan Konfigurasi

Zona waktu utama yang dipakai adalah Asia/Jakarta / WIB. Timestamp mentah diaudit lebih dulu; jika offset eksplisit ada, offset dipertahankan dan dikonversi. Jika timestamp naive, nilai dilokalisasi dengan TIMESTAMP_SOURCE_TIMEZONE.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import re
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency


BASE_DIR = Path(".").resolve()
TABLE_DIR = BASE_DIR / "output" / "rm2_temporal" / "tables"
VIS_DIR = BASE_DIR / "output" / "rm2_temporal" / "visualisasi"
GEPHI_DIR = BASE_DIR / "output" / "rm2_temporal" / "gephi"

PRE_IMPLEMENTATION_HEAD = "901784e2f082482bfc25ebfa19655e888a612da8"
PRE_IMPLEMENTATION_MODIFIED_FILES = ["HCC.gephi"]

TIMESTAMP_SOURCE_TIMEZONE = "UTC"
ANALYSIS_TIMEZONE = "Asia/Jakarta"
TIMEZONE_SENSITIVITY = ["Asia/Jakarta", "Asia/Makassar"]
TEMPORAL_MIN_DATE = None
TEMPORAL_MAX_DATE = None
NOTEBOOK_EXECUTION_TIMESTAMP = pd.Timestamp("2026-07-19 00:00:00", tz=ANALYSIS_TIMEZONE)

WEEKDAY_ORDER = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
WEEKDAY_NUMBER_TO_NAME = dict(enumerate(WEEKDAY_ORDER, start=1))
WEEKDAY_NAME_TO_NUMBER = {name: number for number, name in WEEKDAY_NUMBER_TO_NAME.items()}
DAYPART_ORDER = ["Dini Hari", "Pagi", "Siang", "Sore", "Malam"]
BRAND_ORDER = [
    "Azarine",
    "Daviena",
    "Maryame",
    "The Originote",
    "Mixed_2_Brands",
    "Mixed_3plus_Brands",
    "Not identified",
]
INDIVIDUAL_BRANDS = ["Azarine", "Daviena", "Maryame", "The Originote"]
BRAND_SCORE_COLUMNS = {
    "Azarine": "azarine_score",
    "Daviena": "daviena_score",
    "Maryame": "maryame_score",
    "The Originote": "the_originote_score",
}
BRAND_USER_COLUMNS = {
    "Azarine": "azarine_users",
    "Daviena": "daviena_users",
    "Maryame": "maryame_users",
    "The Originote": "the_originote_users",
}
BRAND_VIDEO_COLUMNS = {
    "Azarine": "azarine_videos",
    "Daviena": "daviena_videos",
    "Maryame": "maryame_videos",
    "The Originote": "the_originote_videos",
}
ACTOR_TYPE_ORDER = ["Individual Actor", "Community Actor", "Mass Actor"]
SENTIMENT_ORDER = ["Positive", "Neutral", "Negative"]
MIN_COMMENTS_TEMPORAL_PROFILE = 10
MIN_ACTIVE_WEEKS = 3
DOMINANT_SHARE_THRESHOLD_RM1 = 0.60

INPUT_PATHS = {
    "dataset": BASE_DIR / "dataset.csv",
    "video_metadata": BASE_DIR / "video_metadata_clean.csv",
    "gephi_hcc_nodes": BASE_DIR / "output" / "gephi" / "gephi_hcc_nodes.csv",
    "gephi_hcc_edges": BASE_DIR / "output" / "gephi" / "gephi_hcc_edges.csv",
    "comment_sentiment": BASE_DIR / "output" / "rm2_sentiment" / "tables" / "comment_sentiment.csv",
    "hcc_sentiment_goals_summary": BASE_DIR / "output" / "rm2_sentiment" / "tables" / "hcc_sentiment_goals_summary.csv",
    "account_actor_type": BASE_DIR / "output" / "rm2_actor_type" / "tables" / "account_actor_type.csv",
    "individual_hcc_association": BASE_DIR / "output" / "rm2_actor_type" / "tables" / "individual_hcc_association.csv",
    "hcc_brand_profile_auto": BASE_DIR / "output" / "tables" / "hcc_brand_profile_auto.csv",
    "community_actor_summary_optional": BASE_DIR / "output" / "rm2_actor_type" / "tables" / "community_actor_summary.csv",
    "three_dimensions_typology_hcc_optional": BASE_DIR / "output" / "rm2_actor_type" / "tables" / "three_dimensions_typology_hcc.csv",
    "mass_actor_exposure_optional": BASE_DIR / "output" / "rm2_actor_type" / "tables" / "mass_actor_exposure.csv",
    "gephi_hcc_nodes_actor_type": BASE_DIR / "output" / "rm2_actor_type" / "gephi" / "gephi_hcc_nodes_actor_type.csv",
    "gephi_hcc_edges_actor_type": BASE_DIR / "output" / "rm2_actor_type" / "gephi" / "gephi_hcc_edges_actor_type.csv",
    "gephi_hcc_nodes_sentiment": BASE_DIR / "output" / "rm2_sentiment" / "gephi" / "gephi_hcc_nodes_sentiment.csv",
    "gephi_hcc_edges_sentiment": BASE_DIR / "output" / "rm2_sentiment" / "gephi" / "gephi_hcc_edges_sentiment.csv",
}

ID_DTYPE = {
    "comment_id": "string",
    "parent_comment_id": "string",
    "user_id": "string",
    "video_id": "string",
    "username": "string",
    "id": "string",
    "label": "string",
    "source": "string",
    "target": "string",
    "community": "string",
    "hcc_id": "string",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_safe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=ID_DTYPE, low_memory=False)


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8")


def clean_identifier(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in {"", "<NA>", "nan", "None", "NaN"}:
        return np.nan
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text


def normalize_identifier_column(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        if column in df.columns:
            df[column] = df[column].map(clean_identifier)
    return df


def safe_divide(numerator, denominator):
    if pd.isna(denominator) or denominator == 0:
        return np.nan
    return numerator / denominator


def ordered_mode(values, order=None, default="Not Observed"):
    series = pd.Series(values).dropna().astype(str)
    series = series[~series.isin(["nan", "None", "<NA>", ""])]
    if series.empty:
        return default
    counts = series.value_counts()
    max_count = counts.max()
    if order is not None:
        for item in order:
            if counts.get(item, 0) == max_count:
                return item
    return sorted(counts[counts == max_count].index)[0]


def peak_hour(values):
    series = pd.Series(values).dropna()
    if series.empty:
        return np.nan
    counts = series.astype(int).value_counts()
    max_count = counts.max()
    return int(sorted(counts[counts == max_count].index)[0])


def normalized_entropy_from_counts(counts) -> float:
    arr = np.asarray(counts, dtype=float)
    total = arr.sum()
    if total <= 0:
        return np.nan
    positive = arr[arr > 0] / total
    return float(-(positive * np.log(positive)).sum() / np.log(len(arr)))


def temporal_reliability(n_valid_timestamp_comments: int, n_active_weeks: int) -> str:
    if n_valid_timestamp_comments < MIN_COMMENTS_TEMPORAL_PROFILE:
        return "Insufficient observations"
    if n_active_weeks < MIN_ACTIVE_WEEKS:
        return "Limited recurrence"
    return "Descriptively sufficient"


def concentration_level(score, n_valid_timestamp_comments: int, n_active_weeks: int) -> str:
    if n_valid_timestamp_comments < MIN_COMMENTS_TEMPORAL_PROFILE or n_active_weeks < MIN_ACTIVE_WEEKS or pd.isna(score):
        return "Insufficient"
    if score < 0.20:
        return "Low weekday concentration"
    if score < 0.40:
        return "Moderate weekday concentration"
    return "High weekday concentration"


def daypart_from_hour(hour):
    if pd.isna(hour):
        return np.nan
    hour = int(hour)
    if 0 <= hour <= 4:
        return "Dini Hari"
    if 5 <= hour <= 10:
        return "Pagi"
    if 11 <= hour <= 14:
        return "Siang"
    if 15 <= hour <= 17:
        return "Sore"
    if 18 <= hour <= 23:
        return "Malam"
    return np.nan


def first_existing(paths: list[Path]) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError("No source file found in candidate paths.")


def weekday_count_columns(prefix: str) -> list[str]:
    return [f"{prefix}_{day}" for day in WEEKDAY_ORDER]


def public_actor_identifier(row: pd.Series) -> str:
    actor_type = row.get("actor_type_primary")
    if pd.isna(actor_type):
        actor_type = ""
    if actor_type == "Mass Actor":
        value = row.get("actor_id_anonymous")
    elif actor_type == "Individual Actor":
        value = row.get("display_name") if pd.notna(row.get("display_name")) else row.get("username")
    else:
        value = row.get("username")
    return clean_identifier(value) or "Not identified"


def format_pct(value, decimals=1):
    if pd.isna(value):
        return "NA"
    return f"{value * 100:.{decimals}f}%"


def sorted_hcc_ids(values) -> list[str]:
    unique_values = sorted(
        {str(clean_identifier(value)) for value in values if pd.notna(clean_identifier(value))},
        key=lambda x: int(x) if str(x).isdigit() else str(x),
    )
    return unique_values


def join_hcc_ids(values) -> str:
    return ";".join(sorted_hcc_ids(values))


def parse_active_brands(row: pd.Series) -> list[str]:
    combo = row.get("brand_combo")
    label = row.get("brand_label_auto")
    primary = row.get("primary_brand")
    if pd.notna(combo) and str(combo).strip():
        parts = [part.strip() for part in str(combo).split("+")]
        return [brand for brand in INDIVIDUAL_BRANDS if brand in parts]
    if label in INDIVIDUAL_BRANDS:
        return [label]
    if primary in INDIVIDUAL_BRANDS and label in INDIVIDUAL_BRANDS:
        return [primary]
    return []


def expected_active_brand_count(label) -> str:
    if label in INDIVIDUAL_BRANDS:
        return "1"
    if label == "Mixed_2_Brands":
        return "2"
    if label == "Mixed_3plus_Brands":
        return ">=3"
    if label == "Not identified":
        return "0"
    return "Unknown"


def brand_membership_validation_status(label, observed_count: int) -> tuple[str, str]:
    if label in INDIVIDUAL_BRANDS and observed_count == 1:
        return "Valid", "Single-brand HCC has one active brand."
    if label == "Mixed_2_Brands" and observed_count == 2:
        return "Valid", "Mixed_2_Brands has two active brands."
    if label == "Mixed_3plus_Brands" and observed_count >= 3:
        return "Valid", "Mixed_3plus_Brands has at least three active brands."
    if label == "Not identified" and observed_count == 0:
        return "Valid", "Not identified has no active brand."
    return "brand_membership_inconsistency", "Observed active-brand count does not match brand_label_auto rule."


def brand_scope_from_label(label: str) -> str:
    if label in INDIVIDUAL_BRANDS:
        return "Single-brand HCC"
    if label in ["Mixed_2_Brands", "Mixed_3plus_Brands"]:
        return "Cross-brand HCC"
    return "Unidentified HCC"


def distribution_string(values) -> str:
    counts = pd.Series(values).dropna().astype(str).value_counts()
    if counts.empty:
        return ""
    return "; ".join([f"{label}={int(count)}" for label, count in counts.sort_index().items()])


def git_command(args: list[str]) -> str:
    result = subprocess.run(["git"] + args, cwd=BASE_DIR, capture_output=True, text=True, check=False)
    return result.stdout.strip() if result.stdout.strip() else result.stderr.strip()


def save_figure(fig, filename: str):
    fig.tight_layout()
    fig.savefig(VIS_DIR / filename, bbox_inches="tight")
    plt.close(fig)


def run_pipeline():
    pd.set_option("display.max_columns", 120)
    sns.set_theme(style="whitegrid", font_scale=0.9)
    plt.rcParams["figure.dpi"] = 120
    plt.rcParams["savefig.dpi"] = 180
    for directory in [TABLE_DIR, VIS_DIR, GEPHI_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

    repo_audit = pd.DataFrame(
        [
            {"field": "pre_implementation_head", "value": PRE_IMPLEMENTATION_HEAD},
            {"field": "pre_implementation_modified_files", "value": "; ".join(PRE_IMPLEMENTATION_MODIFIED_FILES)},
            {"field": "notebook_execution_head", "value": git_command(["rev-parse", "HEAD"])},
            {"field": "notebook_execution_git_status_short", "value": git_command(["status", "--short"])},
        ]
    )
    write_csv(repo_audit, TABLE_DIR / "temporal_repository_audit.csv")

    loaded = {}
    audit_rows = []
    input_checksums_before = {}
    for name, path in INPUT_PATHS.items():
        exists = path.exists()
        row = {"input_name": name, "path": str(path.relative_to(BASE_DIR)), "exists": exists}
        if exists:
            checksum = sha256_file(path)
            input_checksums_before[str(path)] = checksum
            df = read_csv_safe(path)
            loaded[name] = df
            row.update(
                {
                    "sha256": checksum,
                    "n_rows": len(df),
                    "n_columns": len(df.columns),
                    "columns": json.dumps(list(df.columns), ensure_ascii=False),
                }
            )
        else:
            row.update({"sha256": np.nan, "n_rows": np.nan, "n_columns": np.nan, "columns": np.nan})
        audit_rows.append(row)
    input_file_audit = pd.DataFrame(audit_rows)
    write_csv(input_file_audit, TABLE_DIR / "input_file_audit.csv")

    dataset = loaded["dataset"].copy()
    video_metadata = loaded["video_metadata"].copy()
    gephi_hcc_nodes = loaded["gephi_hcc_nodes"].copy()
    gephi_hcc_edges = loaded["gephi_hcc_edges"].copy()
    comment_sentiment = loaded["comment_sentiment"].copy()
    hcc_sentiment_goals = loaded["hcc_sentiment_goals_summary"].copy()
    account_actor_type = loaded["account_actor_type"].copy()
    individual_hcc_association = loaded["individual_hcc_association"].copy()
    hcc_brand_profile = loaded["hcc_brand_profile_auto"].copy()

    for frame in [
        dataset,
        video_metadata,
        gephi_hcc_nodes,
        gephi_hcc_edges,
        comment_sentiment,
        hcc_sentiment_goals,
        account_actor_type,
        individual_hcc_association,
        hcc_brand_profile,
    ]:
        normalize_identifier_column(
            frame,
            ["comment_id", "parent_comment_id", "user_id", "video_id", "username", "id", "label", "source", "target", "community", "hcc_id", "creator"],
        )

    account_actor_type["actor_type_primary"] = pd.Categorical(
        account_actor_type["actor_type_primary"], categories=ACTOR_TYPE_ORDER, ordered=True
    )
    actor_type_counts = account_actor_type["actor_type_primary"].value_counts().reindex(ACTOR_TYPE_ORDER, fill_value=0)
    valid_hcc_ids = set(gephi_hcc_nodes["community"].dropna().map(clean_identifier).astype(str))
    assert "Non-HCC" not in valid_hcc_ids

    baseline_assertions = pd.DataFrame(
        [
            {"baseline": "comments", "expected": 33847, "actual": len(dataset)},
            {"baseline": "videos_metadata_rows", "expected": 55, "actual": len(video_metadata)},
            {"baseline": "Individual Actor accounts", "expected": 43, "actual": int(actor_type_counts.get("Individual Actor", 0))},
            {"baseline": "Community Actor accounts", "expected": 218, "actual": int(actor_type_counts.get("Community Actor", 0))},
            {"baseline": "Mass Actor accounts", "expected": 26166, "actual": int(actor_type_counts.get("Mass Actor", 0))},
            {"baseline": "HCC communities", "expected": 42, "actual": len(valid_hcc_ids)},
            {"baseline": "HCC edges", "expected": 464, "actual": len(gephi_hcc_edges)},
        ]
    )
    baseline_assertions["passed"] = baseline_assertions["expected"] == baseline_assertions["actual"]
    write_csv(baseline_assertions, TABLE_DIR / "baseline_assertions.csv")
    assert baseline_assertions["passed"].all(), baseline_assertions

    hcc_member_map = gephi_hcc_nodes[["id", "community"]].rename(columns={"id": "username", "community": "hcc_id"}).copy()
    hcc_member_map["username"] = hcc_member_map["username"].map(clean_identifier)
    hcc_member_map["hcc_id"] = hcc_member_map["hcc_id"].map(clean_identifier)
    assert set(hcc_member_map["hcc_id"].dropna().astype(str)).issubset(valid_hcc_ids)
    community_actor_accounts = set(
        account_actor_type.loc[account_actor_type["actor_type_primary"].astype(str) == "Community Actor", "username"]
        .dropna()
        .astype(str)
    )
    hcc_node_accounts = set(hcc_member_map["username"].dropna().astype(str))
    community_mapping_validation = pd.DataFrame(
        [
            {"metric": "hcc_node_accounts", "value": len(hcc_node_accounts)},
            {"metric": "community_actor_accounts_from_actor_type", "value": len(community_actor_accounts)},
            {"metric": "accounts_in_hcc_nodes_not_actor_type_community", "value": len(hcc_node_accounts - community_actor_accounts)},
            {"metric": "community_actor_accounts_not_in_hcc_nodes", "value": len(community_actor_accounts - hcc_node_accounts)},
        ]
    )
    write_csv(community_mapping_validation, TABLE_DIR / "community_hcc_mapping_validation.csv")
    assert len(hcc_node_accounts - community_actor_accounts) == 0
    assert len(community_actor_accounts - hcc_node_accounts) == 0

    sentiment_cols = [column for column in ["comment_id", "sentiment_label", "sentiment_score", "sentiment_confidence", "clean_text_light"] if column in comment_sentiment.columns]
    account_cols = [
        "username",
        "actor_id_anonymous",
        "display_name",
        "actor_type_primary",
        "individual_subtype",
        "is_video_creator",
        "is_verified_registry_actor",
        "individual_identification_source",
        "individual_verification_level",
        "is_hcc_member",
        "individual_in_hcc",
        "community",
        "is_lcn_member",
        "mass_position_secondary",
    ]
    account_cols = [column for column in account_cols if column in account_actor_type.columns]
    account_for_comments = account_actor_type[account_cols].copy()
    account_for_comments["comment_username_key"] = account_for_comments["display_name"].map(clean_identifier)
    account_for_comments["comment_username_key"] = account_for_comments["comment_username_key"].combine_first(
        account_for_comments["username"].map(clean_identifier)
    )
    account_for_comments = account_for_comments.rename(columns={"username": "actor_table_username"})
    assert not account_for_comments["comment_username_key"].duplicated().any()
    comments = dataset.merge(comment_sentiment[sentiment_cols], on="comment_id", how="left", validate="one_to_one")
    comments = comments.merge(
        account_for_comments,
        left_on="username",
        right_on="comment_username_key",
        how="left",
        validate="many_to_one",
        suffixes=("", "_account"),
    ).drop(columns=["comment_username_key"])
    comments = comments.merge(hcc_member_map, on="username", how="left", validate="many_to_one")
    comments["actor_type_primary"] = comments["actor_type_primary"].astype("string")
    comments["hcc_id"] = comments["hcc_id"].map(clean_identifier)
    comments["is_direct_hcc_member"] = comments["hcc_id"].notna()
    comments["actor_id_anonymous"] = comments["actor_id_anonymous"].map(clean_identifier)
    comments["display_name"] = comments["display_name"].map(clean_identifier)
    comments["public_actor_id"] = comments.apply(public_actor_identifier, axis=1)
    comments_without_actor_type = int(comments["actor_type_primary"].isna().sum())
    community_comments_without_valid_hcc = int(
        ((comments["actor_type_primary"] == "Community Actor") & (~comments["hcc_id"].isin(valid_hcc_ids))).sum()
    )
    non_hcc_in_outputs = int((comments["hcc_id"] == "Non-HCC").sum())
    actor_hcc_validation = pd.DataFrame(
        [
            {"metric": "comments_without_actor_type", "value": comments_without_actor_type},
            {"metric": "community_actor_comments_without_valid_hcc_id", "value": community_comments_without_valid_hcc},
            {"metric": "comments_with_non_hcc_as_hcc_id", "value": non_hcc_in_outputs},
            {"metric": "distinct_actor_types_in_comments", "value": "; ".join(sorted(comments["actor_type_primary"].dropna().unique()))},
        ]
    )
    write_csv(actor_hcc_validation, TABLE_DIR / "actor_hcc_validation.csv")
    assert comments_without_actor_type == 0
    assert community_comments_without_valid_hcc == 0
    assert non_hcc_in_outputs == 0

    for frame in [hcc_sentiment_goals, hcc_brand_profile]:
        if "hcc_id" in frame.columns:
            frame["hcc_id"] = frame["hcc_id"].map(clean_identifier)
    goal_columns = [
        "hcc_id",
        "community_size",
        "n_comments",
        "positive_count",
        "neutral_count",
        "negative_count",
        "positive_ratio",
        "neutral_ratio",
        "negative_ratio",
        "dominant_sentiment",
        "goal_orientation",
        "primary_brand",
        "brand_label_auto",
        "brand_combo",
        "brand_confidence",
        "narrative_similarity_level",
    ]
    hcc_meta = hcc_sentiment_goals[[column for column in goal_columns if column in hcc_sentiment_goals.columns]].copy()
    brand_columns = [column for column in ["hcc_id", "primary_brand", "brand_label_auto", "brand_combo", "brand_confidence"] if column in hcc_brand_profile.columns]
    hcc_brand_small = hcc_brand_profile[brand_columns].copy()
    hcc_meta = hcc_brand_small.merge(hcc_meta, on="hcc_id", how="outer", suffixes=("_brand_profile", ""))
    for base in ["primary_brand", "brand_label_auto", "brand_combo", "brand_confidence"]:
        profile_col = f"{base}_brand_profile"
        if profile_col in hcc_meta.columns:
            hcc_meta[base] = hcc_meta[base].combine_first(hcc_meta[profile_col]) if base in hcc_meta.columns else hcc_meta[profile_col]
            hcc_meta = hcc_meta.drop(columns=[profile_col])

    comments = comments.merge(
        hcc_meta[
            [
                "hcc_id",
                "goal_orientation",
                "dominant_sentiment",
                "positive_ratio",
                "neutral_ratio",
                "negative_ratio",
                "primary_brand",
                "brand_label_auto",
                "narrative_similarity_level",
            ]
        ],
        on="hcc_id",
        how="left",
        suffixes=("", "_hcc"),
    )
    comments = comments.rename(
        columns={
            "goal_orientation": "hcc_goal_orientation",
            "dominant_sentiment": "hcc_dominant_sentiment",
            "positive_ratio": "hcc_positive_ratio",
            "neutral_ratio": "hcc_neutral_ratio",
            "negative_ratio": "hcc_negative_ratio",
            "primary_brand": "hcc_primary_brand",
            "brand_label_auto": "hcc_brand_label_auto",
            "narrative_similarity_level": "hcc_narrative_similarity_level",
        }
    )
    comments["hcc_brand_label_auto"] = comments["hcc_brand_label_auto"].fillna("Not identified")
    comments["hcc_primary_brand"] = comments["hcc_primary_brand"].fillna("Not identified")
    comments["hcc_goal_orientation"] = comments["hcc_goal_orientation"].fillna("Not Observed")
    comments["sentiment_label"] = comments["sentiment_label"].fillna("Not Available")

    raw_timestamp_examples = comments[["comment_id", "timestamp"]].head(20).copy()
    write_csv(raw_timestamp_examples, TABLE_DIR / "timestamp_raw_examples.csv")
    raw_timestamp = comments["timestamp"].astype("string").str.strip()
    has_explicit_offset = raw_timestamp.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True, na=False)
    aware_timestamp_utc = pd.to_datetime(raw_timestamp.where(has_explicit_offset), errors="coerce", utc=True)
    naive_timestamp = pd.to_datetime(raw_timestamp.where(~has_explicit_offset), errors="coerce")
    naive_timestamp_utc = naive_timestamp.dt.tz_localize(TIMESTAMP_SOURCE_TIMEZONE, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
    timestamp_utc = aware_timestamp_utc.where(has_explicit_offset, naive_timestamp_utc)
    comments["timestamp_has_explicit_offset"] = has_explicit_offset
    comments["timestamp_utc"] = timestamp_utc
    comments["timestamp_local"] = comments["timestamp_utc"].dt.tz_convert(ANALYSIS_TIMEZONE)
    comments["timestamp_wib_sensitivity"] = comments["timestamp_utc"].dt.tz_convert("Asia/Jakarta")
    comments["timestamp_wita_sensitivity"] = comments["timestamp_utc"].dt.tz_convert("Asia/Makassar")
    valid_timestamp_mask = comments["timestamp_utc"].notna()
    weekday_changed_wib_vs_wita = valid_timestamp_mask & (
        comments["timestamp_wib_sensitivity"].dt.dayofweek != comments["timestamp_wita_sensitivity"].dt.dayofweek
    )
    n_comments_weekday_changed_wib_vs_wita = int(weekday_changed_wib_vs_wita.sum())
    weekday_changed_percentage_wib_vs_wita = safe_divide(n_comments_weekday_changed_wib_vs_wita, int(valid_timestamp_mask.sum()))
    timezone_sensitivity_audit = pd.DataFrame(
        [
            {"metric": "timestamp_source_timezone_for_naive_values", "value": TIMESTAMP_SOURCE_TIMEZONE},
            {"metric": "analysis_timezone", "value": ANALYSIS_TIMEZONE},
            {"metric": "timezone_sensitivity", "value": "; ".join(TIMEZONE_SENSITIVITY)},
            {"metric": "total_comments", "value": len(comments)},
            {"metric": "valid_timestamps", "value": int(valid_timestamp_mask.sum())},
            {"metric": "invalid_timestamps", "value": int((~valid_timestamp_mask).sum())},
            {"metric": "timestamp_rows_with_explicit_offset", "value": int(has_explicit_offset.sum())},
            {"metric": "timestamp_rows_without_explicit_offset", "value": int((~has_explicit_offset).sum())},
            {"metric": "timezone_aware_valid_timestamps", "value": int((has_explicit_offset & valid_timestamp_mask).sum())},
            {"metric": "timezone_naive_valid_timestamps", "value": int((~has_explicit_offset & valid_timestamp_mask).sum())},
            {"metric": "n_comments_weekday_changed_wib_vs_wita", "value": n_comments_weekday_changed_wib_vs_wita},
            {"metric": "weekday_changed_percentage_wib_vs_wita", "value": weekday_changed_percentage_wib_vs_wita},
        ]
    )
    write_csv(timezone_sensitivity_audit, TABLE_DIR / "timezone_sensitivity_audit.csv")

    comments["comment_date_local"] = comments["timestamp_local"].dt.date
    comments["comment_year"] = comments["timestamp_local"].dt.year
    comments["comment_month"] = comments["timestamp_local"].dt.month
    comments["comment_year_month"] = comments["timestamp_local"].dt.strftime("%Y-%m")
    comments["iso_week"] = comments["timestamp_local"].dt.isocalendar().week.astype("Int64")
    comments["weekday_number"] = (comments["timestamp_local"].dt.dayofweek + 1).astype("Int64")
    comments["weekday_name_id"] = comments["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    comments["hour_local"] = comments["timestamp_local"].dt.hour.astype("Int64")
    comments["minute_local"] = comments["timestamp_local"].dt.minute.astype("Int64")
    comments["is_weekend"] = comments["weekday_name_id"].isin(["Sabtu", "Minggu"])
    comments["daypart"] = comments["hour_local"].map(daypart_from_hour)
    comments["daypart"] = pd.Categorical(comments["daypart"], categories=DAYPART_ORDER, ordered=True)
    comment_date_ts = pd.to_datetime(comments["comment_date_local"])
    comments["week_start_date"] = (comment_date_ts - pd.to_timedelta(comments["weekday_number"].astype("float") - 1, unit="D")).dt.date
    comments["is_future_timestamp"] = comments["timestamp_local"] > NOTEBOOK_EXECUTION_TIMESTAMP
    outside_min = pd.Series(False, index=comments.index) if TEMPORAL_MIN_DATE is None else comments["timestamp_local"] < pd.Timestamp(TEMPORAL_MIN_DATE, tz=ANALYSIS_TIMEZONE)
    outside_max = pd.Series(False, index=comments.index) if TEMPORAL_MAX_DATE is None else comments["timestamp_local"] > pd.Timestamp(TEMPORAL_MAX_DATE, tz=ANALYSIS_TIMEZONE)
    comments["is_outside_expected_range"] = outside_min | outside_max
    comments["timestamp_quality_status"] = np.select(
        [comments["timestamp_utc"].isna(), comments["is_future_timestamp"], comments["is_outside_expected_range"]],
        ["Invalid timestamp", "Future timestamp", "Outside expected range"],
        default="Valid timestamp",
    )
    quality_flag_columns = [
        "comment_id",
        "public_actor_id",
        "actor_type_primary",
        "video_id",
        "timestamp",
        "timestamp_utc",
        "timestamp_local",
        "timestamp_has_explicit_offset",
        "is_future_timestamp",
        "is_outside_expected_range",
        "timestamp_quality_status",
        "weekday_name_id",
        "hour_local",
        "daypart",
    ]
    write_csv(comments[quality_flag_columns], TABLE_DIR / "comment_temporal_quality_flags.csv")

    valid_comments = comments[comments["timestamp_utc"].notna()].copy()
    community_valid = valid_comments[valid_comments["actor_type_primary"] == "Community Actor"].copy()
    temporal_data_quality_audit = pd.DataFrame(
        [
            {"metric": "total_comments", "value": len(comments)},
            {"metric": "comments_with_valid_timestamp", "value": int(comments["timestamp_utc"].notna().sum())},
            {"metric": "comments_with_invalid_timestamp", "value": int(comments["timestamp_utc"].isna().sum())},
            {"metric": "timestamp_coverage_percentage", "value": safe_divide(int(comments["timestamp_utc"].notna().sum()), len(comments))},
            {"metric": "minimum_timestamp_utc", "value": str(comments["timestamp_utc"].min())},
            {"metric": "maximum_timestamp_utc", "value": str(comments["timestamp_utc"].max())},
            {"metric": "minimum_timestamp_local", "value": str(comments["timestamp_local"].min())},
            {"metric": "maximum_timestamp_local", "value": str(comments["timestamp_local"].max())},
            {"metric": "future_timestamps_relative_to_notebook_execution", "value": int(comments["is_future_timestamp"].sum())},
            {"metric": "duplicate_comment_ids", "value": int(comments["comment_id"].duplicated().sum())},
            {"metric": "comments_without_actor_type", "value": comments_without_actor_type},
            {"metric": "comments_without_video_id", "value": int(comments["video_id"].isna().sum())},
            {"metric": "community_actor_comments_without_valid_hcc_id", "value": community_comments_without_valid_hcc},
            {"metric": "timestamps_outside_plausible_research_range", "value": int(comments["is_outside_expected_range"].sum())},
            {"metric": "n_comments_weekday_changed_wib_vs_wita", "value": n_comments_weekday_changed_wib_vs_wita},
            {"metric": "weekday_changed_percentage_wib_vs_wita", "value": weekday_changed_percentage_wib_vs_wita},
            {"metric": "comments_excluded_from_temporal_summaries_due_to_invalid_timestamp", "value": int(comments["timestamp_utc"].isna().sum())},
            {"metric": "comments_excluded_due_to_future_or_outside_range_flags", "value": 0},
        ]
    )
    write_csv(temporal_data_quality_audit, TABLE_DIR / "temporal_data_quality_audit.csv")

    actor_weekday_index = pd.MultiIndex.from_product([ACTOR_TYPE_ORDER, range(1, 8)], names=["actor_type_primary", "weekday_number"])
    base_actor_weekday = (
        valid_comments.groupby(["actor_type_primary", "weekday_number"], observed=False)
        .agg(
            n_comments=("comment_id", "count"),
            n_valid_timestamp_comments=("timestamp_utc", "count"),
            n_unique_actors=("username", "nunique"),
            n_unique_videos=("video_id", "nunique"),
            n_active_hcc=("hcc_id", "nunique"),
        )
        .reindex(actor_weekday_index, fill_value=0)
    )
    sentiment_actor_weekday = (
        valid_comments[valid_comments["sentiment_label"].isin(SENTIMENT_ORDER)]
        .groupby(["actor_type_primary", "weekday_number", "sentiment_label"], observed=False)
        .size()
        .unstack("sentiment_label", fill_value=0)
        .reindex(actor_weekday_index, fill_value=0)
        .rename(columns={"Positive": "positive_count", "Neutral": "neutral_count", "Negative": "negative_count"})
    )
    actor_type_weekday_summary = base_actor_weekday.join(sentiment_actor_weekday).reset_index()
    actor_type_weekday_summary["weekday_name_id"] = actor_type_weekday_summary["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    actor_totals = valid_comments.groupby("actor_type_primary").size().reindex(ACTOR_TYPE_ORDER, fill_value=0)
    actor_type_weekday_summary["comment_share_within_actor_type"] = actor_type_weekday_summary.apply(
        lambda row: safe_divide(row["n_comments"], actor_totals.get(row["actor_type_primary"], 0)), axis=1
    )
    actor_type_weekday_summary["comments_per_active_video"] = actor_type_weekday_summary.apply(lambda row: safe_divide(row["n_comments"], row["n_unique_videos"]), axis=1)
    actor_type_weekday_summary["comments_per_active_actor"] = actor_type_weekday_summary.apply(lambda row: safe_divide(row["n_comments"], row["n_unique_actors"]), axis=1)
    for sentiment in ["positive", "neutral", "negative"]:
        actor_type_weekday_summary[f"{sentiment}_ratio"] = actor_type_weekday_summary.apply(
            lambda row, s=sentiment: safe_divide(row[f"{s}_count"], row["n_comments"]), axis=1
        )
    actor_type_weekday_summary = actor_type_weekday_summary[
        [
            "actor_type_primary",
            "weekday_number",
            "weekday_name_id",
            "n_comments",
            "n_valid_timestamp_comments",
            "comment_share_within_actor_type",
            "n_unique_actors",
            "n_unique_videos",
            "n_active_hcc",
            "comments_per_active_video",
            "comments_per_active_actor",
            "positive_count",
            "neutral_count",
            "negative_count",
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio",
        ]
    ]
    write_csv(actor_type_weekday_summary, TABLE_DIR / "actor_type_weekday_summary.csv")

    community_weekday_index = pd.Index(range(1, 8), name="weekday_number")
    community_base = (
        community_valid.groupby("weekday_number")
        .agg(
            n_comments=("comment_id", "count"),
            n_unique_community_actors=("username", "nunique"),
            n_active_hcc=("hcc_id", "nunique"),
            n_unique_videos=("video_id", "nunique"),
        )
        .reindex(community_weekday_index, fill_value=0)
        .reset_index()
    )
    community_base["weekday_name_id"] = community_base["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    community_total_comments = int(len(community_valid))
    community_base["comment_share"] = community_base["n_comments"].map(lambda value: safe_divide(value, community_total_comments))
    community_base["comments_per_active_video"] = community_base.apply(lambda row: safe_divide(row["n_comments"], row["n_unique_videos"]), axis=1)
    community_base["comments_per_active_hcc"] = community_base.apply(lambda row: safe_divide(row["n_comments"], row["n_active_hcc"]), axis=1)
    community_sentiment = (
        community_valid[community_valid["sentiment_label"].isin(SENTIMENT_ORDER)]
        .groupby(["weekday_number", "sentiment_label"])
        .size()
        .unstack("sentiment_label", fill_value=0)
        .reindex(community_weekday_index, fill_value=0)
        .rename(columns={"Positive": "positive_count", "Neutral": "neutral_count", "Negative": "negative_count"})
        .reset_index()
    )
    community_weekday_summary = community_base.merge(community_sentiment, on="weekday_number", how="left")
    for column in ["positive_count", "neutral_count", "negative_count"]:
        community_weekday_summary[column] = community_weekday_summary[column].fillna(0).astype(int)
    for sentiment in ["positive", "neutral", "negative"]:
        community_weekday_summary[f"{sentiment}_ratio"] = community_weekday_summary.apply(
            lambda row, s=sentiment: safe_divide(row[f"{s}_count"], row["n_comments"]), axis=1
        )
    community_weekday_summary["dominant_sentiment"] = community_weekday_summary.apply(
        lambda row: ordered_mode(np.repeat(SENTIMENT_ORDER, [row["positive_count"], row["neutral_count"], row["negative_count"]]), SENTIMENT_ORDER),
        axis=1,
    )
    weekday_goal_orientation = community_valid.groupby("weekday_number")["hcc_goal_orientation"].apply(lambda s: ordered_mode(s, default="Not Observed"))
    community_weekday_summary["dominant_goal_orientation_context"] = community_weekday_summary["weekday_number"].map(weekday_goal_orientation).fillna("Not Observed")
    community_weekday_summary = community_weekday_summary[
        [
            "weekday_number",
            "weekday_name_id",
            "n_comments",
            "comment_share",
            "n_unique_community_actors",
            "n_active_hcc",
            "n_unique_videos",
            "comments_per_active_video",
            "comments_per_active_hcc",
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio",
            "dominant_sentiment",
            "dominant_goal_orientation_context",
        ]
    ]
    write_csv(community_weekday_summary, TABLE_DIR / "community_weekday_summary.csv")

    all_weekday = (
        valid_comments.groupby("weekday_number")
        .agg(all_comment_count=("comment_id", "count"), active_video_count=("video_id", "nunique"))
        .reindex(community_weekday_index, fill_value=0)
        .reset_index()
    )
    community_weekday_counts = (
        community_valid.groupby("weekday_number")
        .agg(community_comment_count=("comment_id", "count"), community_active_video_count=("video_id", "nunique"))
        .reindex(community_weekday_index, fill_value=0)
        .reset_index()
    )
    community_weekday_activity_lift = all_weekday.merge(community_weekday_counts, on="weekday_number", how="left")
    community_weekday_activity_lift["weekday"] = community_weekday_activity_lift["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    all_valid_timestamp_comments = int(len(valid_comments))
    all_community_comments = int(len(community_valid))
    community_weekday_activity_lift["overall_weekday_share"] = community_weekday_activity_lift["all_comment_count"].map(lambda value: safe_divide(value, all_valid_timestamp_comments))
    community_weekday_activity_lift["community_weekday_share"] = community_weekday_activity_lift["community_comment_count"].map(lambda value: safe_divide(value, all_community_comments))
    community_weekday_activity_lift["weekday_activity_lift"] = community_weekday_activity_lift.apply(lambda row: safe_divide(row["community_weekday_share"], row["overall_weekday_share"]), axis=1)
    community_weekday_activity_lift["community_comment_prevalence"] = community_weekday_activity_lift.apply(lambda row: safe_divide(row["community_comment_count"], row["all_comment_count"]), axis=1)
    community_weekday_activity_lift["community_comments_per_active_video"] = community_weekday_activity_lift.apply(lambda row: safe_divide(row["community_comment_count"], row["community_active_video_count"]), axis=1)
    community_weekday_activity_lift = community_weekday_activity_lift[
        [
            "weekday",
            "weekday_number",
            "all_comment_count",
            "community_comment_count",
            "overall_weekday_share",
            "community_weekday_share",
            "weekday_activity_lift",
            "community_comment_prevalence",
            "active_video_count",
            "community_active_video_count",
            "community_comments_per_active_video",
        ]
    ]
    write_csv(community_weekday_activity_lift, TABLE_DIR / "community_weekday_activity_lift.csv")

    hcc_weekly_recurrence = community_valid.groupby(["hcc_id", "week_start_date"]).agg(
        n_comments=("comment_id", "count"),
        n_active_accounts=("username", "nunique"),
        n_active_videos=("video_id", "nunique"),
    ).reset_index()
    weekly_modes = community_valid.groupby(["hcc_id", "week_start_date"]).agg(
        dominant_weekday_in_week=("weekday_name_id", lambda s: ordered_mode(s, WEEKDAY_ORDER)),
        dominant_daypart_in_week=("daypart", lambda s: ordered_mode(s, DAYPART_ORDER)),
    ).reset_index()
    hcc_weekly_recurrence = hcc_weekly_recurrence.merge(weekly_modes, on=["hcc_id", "week_start_date"], how="left")
    hcc_weekly_recurrence = hcc_weekly_recurrence.sort_values(["hcc_id", "week_start_date"])

    hcc_size_direct = hcc_member_map.groupby("hcc_id")["username"].nunique().rename("community_size_direct").reset_index()
    hcc_catalog = pd.DataFrame({"hcc_id": sorted(valid_hcc_ids, key=lambda x: int(x) if str(x).isdigit() else str(x))})
    hcc_catalog = hcc_catalog.merge(hcc_size_direct, on="hcc_id", how="left")
    hcc_catalog = hcc_catalog.merge(hcc_meta, on="hcc_id", how="left", suffixes=("", "_meta"))
    hcc_catalog["community_size"] = hcc_catalog["community_size"].combine_first(hcc_catalog["community_size_direct"]) if "community_size" in hcc_catalog.columns else hcc_catalog["community_size_direct"]

    hcc_rows = []
    for _, hcc_row in hcc_catalog.iterrows():
        hcc_id = hcc_row["hcc_id"]
        hcc_all = comments[(comments["actor_type_primary"] == "Community Actor") & (comments["hcc_id"] == hcc_id)].copy()
        hcc_valid = community_valid[community_valid["hcc_id"] == hcc_id].copy()
        weekday_counts = hcc_valid["weekday_name_id"].value_counts().reindex(WEEKDAY_ORDER, fill_value=0)
        entropy = normalized_entropy_from_counts(weekday_counts.values)
        concentration_score = np.nan if pd.isna(entropy) else 1 - entropy
        n_active_weeks = int(hcc_valid["week_start_date"].nunique())
        n_valid = int(len(hcc_valid))
        dominant_weekday = ordered_mode(hcc_valid["weekday_name_id"], WEEKDAY_ORDER)
        dominant_count = int(weekday_counts.get(dominant_weekday, 0)) if dominant_weekday in WEEKDAY_ORDER else 0
        dominant_share = safe_divide(dominant_count, n_valid)
        if n_valid == 0:
            second_weekday = "Not Observed"
            second_share = np.nan
        else:
            ordered_weekday_counts = pd.Series({day: weekday_counts[day] for day in WEEKDAY_ORDER})
            top_ordered = sorted(WEEKDAY_ORDER, key=lambda day: (-ordered_weekday_counts[day], WEEKDAY_NAME_TO_NUMBER[day]))
            second_weekday = top_ordered[1] if len(top_ordered) > 1 else "Not Observed"
            second_share = safe_divide(int(weekday_counts.get(second_weekday, 0)), n_valid)
        dominant_daypart = ordered_mode(hcc_valid["daypart"], DAYPART_ORDER)
        hcc_peak_hour = peak_hour(hcc_valid["hour_local"])
        dominant_weekday_active_weeks = int(hcc_valid.loc[hcc_valid["weekday_name_id"] == dominant_weekday, "week_start_date"].nunique()) if dominant_weekday in WEEKDAY_ORDER else 0
        dominant_weekday_week_coverage = safe_divide(dominant_weekday_active_weeks, n_active_weeks)
        reliability = temporal_reliability(n_valid, n_active_weeks)
        row = {
            "hcc_id": hcc_id,
            "community_size": int(hcc_row["community_size"]) if pd.notna(hcc_row.get("community_size")) else np.nan,
            "primary_brand": hcc_row.get("primary_brand", "Not identified"),
            "brand_label_auto": hcc_row.get("brand_label_auto", "Not identified"),
            "goal_orientation": hcc_row.get("goal_orientation", "Not Observed"),
            "dominant_sentiment": hcc_row.get("dominant_sentiment", "Not Observed"),
            "positive_ratio": hcc_row.get("positive_ratio", np.nan),
            "neutral_ratio": hcc_row.get("neutral_ratio", np.nan),
            "negative_ratio": hcc_row.get("negative_ratio", np.nan),
            "narrative_similarity_level": hcc_row.get("narrative_similarity_level", "Not Observed"),
            "n_comments": int(len(hcc_all)),
            "n_valid_timestamp_comments": n_valid,
            "n_unique_commenting_accounts": int(hcc_valid["username"].nunique()),
            "n_active_videos": int(hcc_valid["video_id"].nunique()),
            "n_active_dates": int(hcc_valid["comment_date_local"].nunique()),
            "n_active_weeks": n_active_weeks,
            "dominant_weekday": dominant_weekday,
            "dominant_weekday_comment_count": dominant_count,
            "dominant_weekday_share": dominant_share,
            "second_weekday": second_weekday,
            "second_weekday_share": second_share,
            "weekday_entropy_normalized": entropy,
            "weekday_concentration_score": concentration_score,
            "weekday_concentration_level": concentration_level(concentration_score, n_valid, n_active_weeks),
            "weekend_comment_share": safe_divide(int(hcc_valid["is_weekend"].sum()), n_valid),
            "dominant_daypart": dominant_daypart,
            "peak_hour": hcc_peak_hour,
            "dominant_weekday_active_weeks": dominant_weekday_active_weeks,
            "dominant_weekday_week_coverage": dominant_weekday_week_coverage,
            "temporal_profile_reliability": reliability,
        }
        for day in WEEKDAY_ORDER:
            row[f"comments_{day}"] = int(weekday_counts[day])
            row[f"share_{day}"] = safe_divide(int(weekday_counts[day]), n_valid)
        hcc_rows.append(row)
    hcc_weekday_profile = pd.DataFrame(hcc_rows)
    ordered_hcc_cols = (
        [
            "hcc_id",
            "community_size",
            "primary_brand",
            "brand_label_auto",
            "goal_orientation",
            "dominant_sentiment",
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio",
            "narrative_similarity_level",
            "n_comments",
            "n_valid_timestamp_comments",
            "n_unique_commenting_accounts",
            "n_active_videos",
            "n_active_dates",
            "n_active_weeks",
        ]
        + weekday_count_columns("comments")
        + weekday_count_columns("share")
        + [
            "dominant_weekday",
            "dominant_weekday_comment_count",
            "dominant_weekday_share",
            "second_weekday",
            "second_weekday_share",
            "weekday_entropy_normalized",
            "weekday_concentration_score",
            "weekday_concentration_level",
            "weekend_comment_share",
            "dominant_daypart",
            "peak_hour",
            "dominant_weekday_active_weeks",
            "dominant_weekday_week_coverage",
            "temporal_profile_reliability",
        ]
    )
    hcc_weekday_profile = hcc_weekday_profile[ordered_hcc_cols]
    write_csv(hcc_weekday_profile, TABLE_DIR / "hcc_weekday_profile.csv")
    hcc_weekly_recurrence = hcc_weekly_recurrence.merge(hcc_weekday_profile[["hcc_id", "dominant_weekday"]], on="hcc_id", how="left")
    hcc_weekly_recurrence["contains_overall_dominant_weekday"] = hcc_weekly_recurrence["dominant_weekday_in_week"] == hcc_weekly_recurrence["dominant_weekday"]
    write_csv(hcc_weekly_recurrence, TABLE_DIR / "hcc_weekly_recurrence.csv")

    actor_hour_index = pd.MultiIndex.from_product([ACTOR_TYPE_ORDER, range(1, 8), range(24)], names=["actor_type_primary", "weekday_number", "hour_local"])
    actor_type_weekday_hour_matrix = (
        valid_comments.groupby(["actor_type_primary", "weekday_number", "hour_local"], observed=False)
        .agg(n_comments=("comment_id", "count"), n_unique_actors=("username", "nunique"), n_unique_videos=("video_id", "nunique"))
        .reindex(actor_hour_index, fill_value=0)
        .reset_index()
    )
    actor_type_weekday_hour_matrix["weekday_name_id"] = actor_type_weekday_hour_matrix["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    actor_type_weekday_hour_matrix["comment_share_within_actor_type"] = actor_type_weekday_hour_matrix.apply(lambda row: safe_divide(row["n_comments"], actor_totals.get(row["actor_type_primary"], 0)), axis=1)
    actor_type_weekday_hour_matrix = actor_type_weekday_hour_matrix[
        ["actor_type_primary", "weekday_name_id", "weekday_number", "hour_local", "n_comments", "n_unique_actors", "n_unique_videos", "comment_share_within_actor_type"]
    ]
    write_csv(actor_type_weekday_hour_matrix, TABLE_DIR / "actor_type_weekday_hour_matrix.csv")

    community_hour_index = pd.MultiIndex.from_product([range(1, 8), range(24)], names=["weekday_number", "hour_local"])
    community_weekday_hour_matrix = (
        community_valid.groupby(["weekday_number", "hour_local"])
        .agg(n_comments=("comment_id", "count"), n_unique_community_actors=("username", "nunique"), n_unique_videos=("video_id", "nunique"), n_active_hcc=("hcc_id", "nunique"))
        .reindex(community_hour_index, fill_value=0)
        .reset_index()
    )
    community_weekday_hour_matrix["weekday_name_id"] = community_weekday_hour_matrix["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    community_weekday_hour_matrix["comment_share"] = community_weekday_hour_matrix["n_comments"].map(lambda value: safe_divide(value, community_total_comments))
    community_weekday_hour_matrix = community_weekday_hour_matrix[
        ["weekday_name_id", "weekday_number", "hour_local", "n_comments", "n_unique_community_actors", "n_unique_videos", "n_active_hcc", "comment_share"]
    ]
    write_csv(community_weekday_hour_matrix, TABLE_DIR / "community_weekday_hour_matrix.csv")
    eligible_hcc_ids = hcc_weekday_profile.loc[hcc_weekday_profile["n_valid_timestamp_comments"] >= MIN_COMMENTS_TEMPORAL_PROFILE, "hcc_id"].tolist()
    hcc_hour_index = pd.MultiIndex.from_product([eligible_hcc_ids, range(1, 8), range(24)], names=["hcc_id", "weekday_number", "hour_local"])
    hcc_weekday_hour_matrix = (
        community_valid[community_valid["hcc_id"].isin(eligible_hcc_ids)]
        .groupby(["hcc_id", "weekday_number", "hour_local"])
        .agg(n_comments=("comment_id", "count"), n_unique_community_actors=("username", "nunique"), n_unique_videos=("video_id", "nunique"))
        .reindex(hcc_hour_index, fill_value=0)
        .reset_index()
    )
    hcc_totals = hcc_weekday_profile.set_index("hcc_id")["n_valid_timestamp_comments"]
    hcc_weekday_hour_matrix["weekday_name_id"] = hcc_weekday_hour_matrix["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    hcc_weekday_hour_matrix["comment_share_within_hcc"] = hcc_weekday_hour_matrix.apply(lambda row: safe_divide(row["n_comments"], hcc_totals.get(row["hcc_id"], 0)), axis=1)
    hcc_weekday_hour_matrix = hcc_weekday_hour_matrix[
        ["hcc_id", "weekday_name_id", "weekday_number", "hour_local", "n_comments", "n_unique_community_actors", "n_unique_videos", "comment_share_within_hcc"]
    ]
    write_csv(hcc_weekday_hour_matrix, TABLE_DIR / "hcc_weekday_hour_matrix.csv")

    actor_daypart_index = pd.MultiIndex.from_product([ACTOR_TYPE_ORDER, DAYPART_ORDER], names=["actor_type_primary", "daypart"])
    actor_type_daypart_summary = (
        valid_comments.groupby(["actor_type_primary", "daypart"], observed=False)
        .agg(n_comments=("comment_id", "count"), n_unique_actors=("username", "nunique"), n_unique_videos=("video_id", "nunique"))
        .reindex(actor_daypart_index, fill_value=0)
        .reset_index()
    )
    actor_type_daypart_summary["comment_share_within_actor_type"] = actor_type_daypart_summary.apply(lambda row: safe_divide(row["n_comments"], actor_totals.get(row["actor_type_primary"], 0)), axis=1)
    actor_daypart_modes = valid_comments.groupby("actor_type_primary")["daypart"].apply(lambda s: ordered_mode(s, DAYPART_ORDER))
    actor_type_daypart_summary["dominant_daypart"] = actor_type_daypart_summary["actor_type_primary"].map(actor_daypart_modes)
    write_csv(actor_type_daypart_summary, TABLE_DIR / "actor_type_daypart_summary.csv")

    hcc_daypart_index = pd.MultiIndex.from_product([hcc_catalog["hcc_id"].tolist(), DAYPART_ORDER], names=["hcc_id", "daypart"])
    hcc_daypart_summary = (
        community_valid.groupby(["hcc_id", "daypart"], observed=False)
        .agg(n_comments=("comment_id", "count"), n_unique_community_actors=("username", "nunique"), n_unique_videos=("video_id", "nunique"))
        .reindex(hcc_daypart_index, fill_value=0)
        .reset_index()
    )
    hcc_daypart_summary = hcc_daypart_summary.merge(
        hcc_weekday_profile[["hcc_id", "brand_label_auto", "goal_orientation", "n_valid_timestamp_comments", "dominant_daypart", "temporal_profile_reliability"]],
        on="hcc_id",
        how="left",
    )
    hcc_daypart_summary["comment_share_within_hcc"] = hcc_daypart_summary.apply(lambda row: safe_divide(row["n_comments"], row["n_valid_timestamp_comments"]), axis=1)
    write_csv(hcc_daypart_summary, TABLE_DIR / "hcc_daypart_summary.csv")

    community_valid["brand_label_auto_context"] = community_valid["hcc_brand_label_auto"].fillna("Not identified").replace({"Non-HCC": "Not identified"})
    brand_daypart_index = pd.MultiIndex.from_product([BRAND_ORDER, DAYPART_ORDER], names=["brand_label_auto", "daypart"])
    brand_daypart_summary = (
        community_valid.groupby(["brand_label_auto_context", "daypart"], observed=False)
        .agg(n_comments=("comment_id", "count"), n_community_actors=("username", "nunique"), n_unique_videos=("video_id", "nunique"), n_hcc=("hcc_id", "nunique"))
        .rename_axis(index={"brand_label_auto_context": "brand_label_auto"})
        .reindex(brand_daypart_index, fill_value=0)
        .reset_index()
    )
    brand_totals = community_valid.groupby("brand_label_auto_context").size().reindex(BRAND_ORDER, fill_value=0)
    brand_daypart_modes = community_valid.groupby("brand_label_auto_context")["daypart"].apply(lambda s: ordered_mode(s, DAYPART_ORDER)).reindex(BRAND_ORDER).fillna("Not Observed")
    brand_daypart_summary["comment_share_within_brand"] = brand_daypart_summary.apply(lambda row: safe_divide(row["n_comments"], brand_totals.get(row["brand_label_auto"], 0)), axis=1)
    brand_daypart_summary["dominant_daypart"] = brand_daypart_summary["brand_label_auto"].map(brand_daypart_modes)
    write_csv(brand_daypart_summary, TABLE_DIR / "brand_daypart_summary.csv")

    # HCC brand context summaries. Brand labels come only from RM1 auto-labeling
    # based on video metadata hashtags.
    community_all = comments[comments["actor_type_primary"] == "Community Actor"].copy()
    hcc_comment_stats_all = (
        community_all.groupby("hcc_id")
        .agg(
            n_comments=("comment_id", "count"),
            n_unique_videos=("video_id", "nunique"),
            n_community_actors=("username", "nunique"),
        )
        .reset_index()
    )
    hcc_brand_base = hcc_brand_profile.copy()
    hcc_brand_base["hcc_id"] = hcc_brand_base["hcc_id"].map(clean_identifier)
    hcc_brand_base = hcc_brand_base[hcc_brand_base["hcc_id"].isin(valid_hcc_ids)].copy()
    hcc_brand_base["brand_label_auto"] = hcc_brand_base["brand_label_auto"].fillna("Not identified")
    hcc_brand_base["brand_confidence"] = hcc_brand_base["brand_confidence"].fillna("None")
    hcc_brand_base["brand_scope"] = hcc_brand_base["brand_label_auto"].map(brand_scope_from_label)
    hcc_brand_base = hcc_brand_base.merge(hcc_comment_stats_all, on="hcc_id", how="left")
    hcc_brand_base = hcc_brand_base.merge(
        hcc_sentiment_goals[
            [
                "hcc_id",
                "goal_orientation",
                "dominant_sentiment",
                "positive_ratio",
                "neutral_ratio",
                "negative_ratio",
                "narrative_similarity_level",
            ]
        ],
        on="hcc_id",
        how="left",
    )
    for column in ["n_comments", "n_unique_videos", "n_community_actors"]:
        hcc_brand_base[column] = hcc_brand_base[column].fillna(0).astype(int)
    assert hcc_brand_base["hcc_id"].nunique() == len(valid_hcc_ids)
    assert not hcc_brand_base["hcc_id"].duplicated().any()

    total_valid_hcc = int(hcc_brand_base["hcc_id"].nunique())
    exclusive_rows = []
    for brand_label in BRAND_ORDER:
        subset = hcc_brand_base[hcc_brand_base["brand_label_auto"] == brand_label]
        n_hcc = int(subset["hcc_id"].nunique())
        exclusive_rows.append(
            {
                "brand_label_auto": brand_label,
                "classification_approach": "Exclusive HCC Brand Classification",
                "n_hcc": n_hcc,
                "hcc_percentage": safe_divide(n_hcc, total_valid_hcc) * 100,
                "n_community_actors": int(subset["community_size"].sum()),
                "community_actor_percentage": safe_divide(int(subset["community_size"].sum()), int(hcc_brand_base["community_size"].sum())) * 100,
                "n_comments": int(subset["n_comments"].sum()),
                "n_unique_videos": int(
                    community_all[community_all["hcc_id"].isin(subset["hcc_id"])]["video_id"].nunique()
                ),
                "mean_community_size": subset["community_size"].mean(),
                "median_community_size": subset["community_size"].median(),
                "min_community_size": subset["community_size"].min(),
                "max_community_size": subset["community_size"].max(),
                "hcc_ids": join_hcc_ids(subset["hcc_id"]),
            }
        )
    hcc_brand_exclusive_summary = pd.DataFrame(exclusive_rows)
    total_row = {
        "brand_label_auto": "Total",
        "classification_approach": "Exclusive HCC Brand Classification",
        "n_hcc": total_valid_hcc,
        "hcc_percentage": 100.0,
        "n_community_actors": int(hcc_brand_base["community_size"].sum()),
        "community_actor_percentage": 100.0,
        "n_comments": int(hcc_brand_base["n_comments"].sum()),
        "n_unique_videos": int(community_all["video_id"].nunique()),
        "mean_community_size": hcc_brand_base["community_size"].mean(),
        "median_community_size": hcc_brand_base["community_size"].median(),
        "min_community_size": hcc_brand_base["community_size"].min(),
        "max_community_size": hcc_brand_base["community_size"].max(),
        "hcc_ids": join_hcc_ids(hcc_brand_base["hcc_id"]),
    }
    hcc_brand_exclusive_summary = pd.concat([hcc_brand_exclusive_summary, pd.DataFrame([total_row])], ignore_index=True)
    assert int(hcc_brand_exclusive_summary.loc[hcc_brand_exclusive_summary["brand_label_auto"] != "Total", "n_hcc"].sum()) == total_valid_hcc
    write_csv(hcc_brand_exclusive_summary, TABLE_DIR / "hcc_brand_exclusive_summary.csv")

    incidence_rows = []
    audit_rows = []
    matrix_rows = []
    for _, row in hcc_brand_base.iterrows():
        active_brands = parse_active_brands(row)
        observed_count = len(set(active_brands))
        validation_status, notes = brand_membership_validation_status(row["brand_label_auto"], observed_count)
        audit_rows.append(
            {
                "hcc_id": row["hcc_id"],
                "brand_label_auto": row["brand_label_auto"],
                "brand_combo": row.get("brand_combo"),
                "expected_active_brand_count": expected_active_brand_count(row["brand_label_auto"]),
                "observed_active_brand_count": observed_count,
                "validation_status": validation_status,
                "notes": notes,
            }
        )
        matrix_row = {
            "hcc_id": row["hcc_id"],
            "primary_brand": row.get("primary_brand"),
            "brand_label_auto": row["brand_label_auto"],
            "n_active_brands": observed_count,
        }
        for brand in INDIVIDUAL_BRANDS:
            matrix_row[brand] = int(brand in active_brands)
            if brand in active_brands:
                incidence_rows.append(
                    {
                        "hcc_id": row["hcc_id"],
                        "brand": brand,
                        "is_active_brand": True,
                        "is_primary_brand": row.get("primary_brand") == brand,
                        "brand_label_auto": row["brand_label_auto"],
                        "brand_combo": row.get("brand_combo"),
                        "brand_confidence": row.get("brand_confidence", "None"),
                        "brand_score": row.get(BRAND_SCORE_COLUMNS[brand], np.nan),
                        "brand_user_count": row.get(BRAND_USER_COLUMNS[brand], np.nan),
                        "brand_video_count": row.get(BRAND_VIDEO_COLUMNS[brand], np.nan),
                        "community_size": row.get("community_size", np.nan),
                        "n_comments": row.get("n_comments", 0),
                        "goal_orientation": row.get("goal_orientation"),
                        "dominant_sentiment": row.get("dominant_sentiment"),
                        "narrative_similarity_level": row.get("narrative_similarity_level"),
                    }
                )
        matrix_rows.append(matrix_row)
    hcc_brand_membership_long = pd.DataFrame(incidence_rows)
    hcc_brand_incidence_matrix = pd.DataFrame(matrix_rows)[
        ["hcc_id"] + INDIVIDUAL_BRANDS + ["n_active_brands", "primary_brand", "brand_label_auto"]
    ]
    hcc_brand_membership_audit = pd.DataFrame(audit_rows)
    write_csv(hcc_brand_membership_long, TABLE_DIR / "hcc_brand_membership_long.csv")
    write_csv(hcc_brand_incidence_matrix, TABLE_DIR / "hcc_brand_incidence_matrix.csv")
    write_csv(hcc_brand_membership_audit, TABLE_DIR / "hcc_brand_membership_audit.csv")
    assert not hcc_brand_membership_long.duplicated(["hcc_id", "brand"]).any()
    assert set(hcc_brand_membership_long["hcc_id"]).issubset(valid_hcc_ids)
    assert (hcc_brand_membership_long["is_active_brand"] == True).all()

    multilabel_rows = []
    for brand in INDIVIDUAL_BRANDS:
        membership_subset = hcc_brand_membership_long[hcc_brand_membership_long["brand"] == brand]
        hcc_ids = sorted_hcc_ids(membership_subset["hcc_id"])
        hcc_subset = hcc_brand_base[hcc_brand_base["hcc_id"].isin(hcc_ids)]
        comments_subset = community_all.merge(
            membership_subset[["hcc_id", "brand"]], on="hcc_id", how="inner"
        )
        n_hcc_associated = int(len(hcc_ids))
        multilabel_rows.append(
            {
                "brand": brand,
                "classification_approach": "Multi-label HCC-Brand Incidence",
                "n_hcc_associated": n_hcc_associated,
                "hcc_incidence_percentage": safe_divide(n_hcc_associated, total_valid_hcc) * 100,
                "n_hcc_as_primary_brand": int((hcc_subset["primary_brand"] == brand).sum()),
                "n_hcc_single_brand": int((hcc_subset["brand_label_auto"] == brand).sum()),
                "n_hcc_from_mixed_2": int((hcc_subset["brand_label_auto"] == "Mixed_2_Brands").sum()),
                "n_hcc_from_mixed_3plus": int((hcc_subset["brand_label_auto"] == "Mixed_3plus_Brands").sum()),
                "n_high_confidence_hcc": int((hcc_subset["brand_confidence"] == "High").sum()),
                "n_medium_confidence_hcc": int((hcc_subset["brand_confidence"] == "Medium").sum()),
                "n_low_confidence_hcc": int((hcc_subset["brand_confidence"] == "Low").sum()),
                "n_none_confidence_hcc": int((hcc_subset["brand_confidence"] == "None").sum()),
                "n_community_actors": int(hcc_subset["community_size"].sum()),
                "n_comments": int(comments_subset["comment_id"].count()),
                "n_unique_videos": int(comments_subset["video_id"].nunique()),
                "mean_community_size": hcc_subset["community_size"].mean(),
                "hcc_ids": join_hcc_ids(hcc_ids),
                "primary_hcc_ids": join_hcc_ids(hcc_subset.loc[hcc_subset["primary_brand"] == brand, "hcc_id"]),
                "mixed_hcc_ids": join_hcc_ids(hcc_subset.loc[hcc_subset["brand_label_auto"].isin(["Mixed_2_Brands", "Mixed_3plus_Brands"]), "hcc_id"]),
            }
        )
    hcc_brand_multilabel_summary = pd.DataFrame(multilabel_rows)
    write_csv(hcc_brand_multilabel_summary, TABLE_DIR / "hcc_brand_multilabel_summary.csv")

    scope_rows = []
    for scope in ["Single-brand HCC", "Cross-brand HCC", "Unidentified HCC"]:
        subset = hcc_brand_base[hcc_brand_base["brand_scope"] == scope]
        hcc_ids = sorted_hcc_ids(subset["hcc_id"])
        comments_subset = community_all[community_all["hcc_id"].isin(hcc_ids)]
        scope_rows.append(
            {
                "brand_scope": scope,
                "interpretation_note": (
                    "Cross-brand HCC menunjukkan bahwa anggota komunitas teramati aktif pada video dengan konteks lebih dari satu brand. "
                    "Temuan tersebut dapat mencerminkan diskusi lintas produk, perbandingan brand, partisipasi umum pada konten skincare, atau amplifikasi lintas konteks. "
                    "Label ini tidak membuktikan bahwa komunitas berafiliasi dengan seluruh brand tersebut."
                    if scope == "Cross-brand HCC"
                    else ""
                ),
                "n_hcc": int(subset["hcc_id"].nunique()),
                "percentage_hcc": safe_divide(int(subset["hcc_id"].nunique()), total_valid_hcc) * 100,
                "n_community_actor": int(subset["community_size"].sum()),
                "n_comments": int(comments_subset["comment_id"].count()),
                "n_unique_videos": int(comments_subset["video_id"].nunique()),
                "mean_hcc_size": subset["community_size"].mean(),
                "mean_positive_ratio": subset["positive_ratio"].mean(),
                "mean_neutral_ratio": subset["neutral_ratio"].mean(),
                "mean_negative_ratio": subset["negative_ratio"].mean(),
                "goal_orientation_distribution": distribution_string(subset["goal_orientation"]),
                "narrative_similarity_level_distribution": distribution_string(subset["narrative_similarity_level"]),
                "hcc_ids": join_hcc_ids(hcc_ids),
            }
        )
    hcc_brand_scope_summary = pd.DataFrame(scope_rows)
    write_csv(hcc_brand_scope_summary, TABLE_DIR / "hcc_brand_scope_summary.csv")

    brand_comment_membership = community_valid.merge(
        hcc_brand_membership_long[["hcc_id", "brand"]], on="hcc_id", how="inner"
    )
    brand_weekday_index = pd.MultiIndex.from_product(
        [INDIVIDUAL_BRANDS, range(1, 8)], names=["brand", "weekday_number"]
    )
    brand_weekday_group = brand_comment_membership.groupby(["brand", "weekday_number"]).agg(
        n_active_hcc=("hcc_id", "nunique"),
        n_community_actors=("username", "nunique"),
        n_comments=("comment_id", "count"),
        n_unique_videos=("video_id", "nunique"),
        hcc_ids=("hcc_id", join_hcc_ids),
    )
    brand_weekday_active_hcc_summary = brand_weekday_group.reindex(brand_weekday_index).reset_index()
    for column in ["n_active_hcc", "n_community_actors", "n_comments", "n_unique_videos"]:
        brand_weekday_active_hcc_summary[column] = brand_weekday_active_hcc_summary[column].fillna(0).astype(int)
    brand_weekday_active_hcc_summary["hcc_ids"] = brand_weekday_active_hcc_summary["hcc_ids"].fillna("")
    associated_hcc_count_by_brand = hcc_brand_multilabel_summary.set_index("brand")["n_hcc_associated"]
    brand_weekday_active_hcc_summary["weekday_name_id"] = brand_weekday_active_hcc_summary["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    brand_weekday_active_hcc_summary["active_hcc_share_within_brand"] = brand_weekday_active_hcc_summary.apply(
        lambda row: safe_divide(row["n_active_hcc"], associated_hcc_count_by_brand.get(row["brand"], 0)), axis=1
    )
    brand_weekday_active_hcc_summary["comments_per_active_hcc"] = brand_weekday_active_hcc_summary.apply(
        lambda row: safe_divide(row["n_comments"], row["n_active_hcc"]), axis=1
    )
    brand_weekday_active_hcc_summary["comments_per_active_video"] = brand_weekday_active_hcc_summary.apply(
        lambda row: safe_divide(row["n_comments"], row["n_unique_videos"]), axis=1
    )
    brand_weekday_sentiment = (
        brand_comment_membership[brand_comment_membership["sentiment_label"].isin(SENTIMENT_ORDER)]
        .groupby(["brand", "weekday_number", "sentiment_label"])
        .size()
        .unstack("sentiment_label", fill_value=0)
        .reindex(brand_weekday_index, fill_value=0)
        .rename(columns={"Positive": "positive_count", "Neutral": "neutral_count", "Negative": "negative_count"})
        .reset_index()
    )
    brand_weekday_active_hcc_summary = brand_weekday_active_hcc_summary.merge(
        brand_weekday_sentiment, on=["brand", "weekday_number"], how="left"
    )
    for sentiment in ["positive", "neutral", "negative"]:
        brand_weekday_active_hcc_summary[f"{sentiment}_ratio"] = brand_weekday_active_hcc_summary.apply(
            lambda row, s=sentiment: safe_divide(row[f"{s}_count"], row["n_comments"]), axis=1
        )
    brand_weekday_active_hcc_summary = brand_weekday_active_hcc_summary[
        [
            "brand",
            "weekday_number",
            "weekday_name_id",
            "n_active_hcc",
            "active_hcc_share_within_brand",
            "n_community_actors",
            "n_comments",
            "n_unique_videos",
            "comments_per_active_hcc",
            "comments_per_active_video",
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio",
            "hcc_ids",
        ]
    ]
    write_csv(brand_weekday_active_hcc_summary, TABLE_DIR / "brand_weekday_active_hcc_summary.csv")

    brand_hour_index = pd.MultiIndex.from_product(
        [INDIVIDUAL_BRANDS, range(1, 8), range(24)], names=["brand", "weekday_number", "hour_local"]
    )
    brand_weekday_hour_active_hcc = (
        brand_comment_membership.groupby(["brand", "weekday_number", "hour_local"])
        .agg(
            n_active_hcc=("hcc_id", "nunique"),
            n_community_actors=("username", "nunique"),
            n_comments=("comment_id", "count"),
            n_unique_videos=("video_id", "nunique"),
            hcc_ids=("hcc_id", join_hcc_ids),
        )
        .reindex(brand_hour_index)
        .reset_index()
    )
    for column in ["n_active_hcc", "n_community_actors", "n_comments", "n_unique_videos"]:
        brand_weekday_hour_active_hcc[column] = brand_weekday_hour_active_hcc[column].fillna(0).astype(int)
    brand_weekday_hour_active_hcc["hcc_ids"] = brand_weekday_hour_active_hcc["hcc_ids"].fillna("")
    brand_weekday_hour_active_hcc["weekday_name_id"] = brand_weekday_hour_active_hcc["weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    brand_weekday_hour_active_hcc = brand_weekday_hour_active_hcc[
        ["brand", "weekday_number", "weekday_name_id", "hour_local", "n_active_hcc", "n_community_actors", "n_comments", "n_unique_videos", "hcc_ids"]
    ]
    write_csv(brand_weekday_hour_active_hcc, TABLE_DIR / "brand_weekday_hour_active_hcc.csv")

    brand_profile_rows = []
    for brand in INDIVIDUAL_BRANDS:
        membership_subset = hcc_brand_membership_long[hcc_brand_membership_long["brand"] == brand]
        hcc_ids = sorted_hcc_ids(membership_subset["hcc_id"])
        hcc_subset = hcc_brand_base[hcc_brand_base["hcc_id"].isin(hcc_ids)]
        comment_subset = brand_comment_membership[brand_comment_membership["brand"] == brand]
        weekday_comment_counts = comment_subset["weekday_name_id"].value_counts().reindex(WEEKDAY_ORDER, fill_value=0)
        weekday_active_hcc_counts = (
            brand_weekday_active_hcc_summary[brand_weekday_active_hcc_summary["brand"] == brand]
            .set_index("weekday_name_id")["n_active_hcc"]
            .reindex(WEEKDAY_ORDER, fill_value=0)
        )
        entropy = normalized_entropy_from_counts(weekday_comment_counts.values)
        concentration_score = np.nan if pd.isna(entropy) else 1 - entropy
        dominant_by_comments = ordered_mode(comment_subset["weekday_name_id"], WEEKDAY_ORDER)
        if weekday_active_hcc_counts.sum() > 0:
            dominant_by_active_hcc = sorted(
                WEEKDAY_ORDER,
                key=lambda day: (-weekday_active_hcc_counts.get(day, 0), WEEKDAY_NAME_TO_NUMBER[day]),
            )[0]
        else:
            dominant_by_active_hcc = "Not Observed"
        brand_profile_rows.append(
            {
                "brand": brand,
                "n_hcc_associated": int(len(hcc_ids)),
                "n_hcc_primary": int((hcc_subset["primary_brand"] == brand).sum()),
                "n_single_brand_hcc": int((hcc_subset["brand_label_auto"] == brand).sum()),
                "n_cross_brand_hcc": int(hcc_subset["brand_label_auto"].isin(["Mixed_2_Brands", "Mixed_3plus_Brands"]).sum()),
                "n_community_actors": int(hcc_subset["community_size"].sum()),
                "n_comments": int(comment_subset["comment_id"].count()),
                "n_unique_videos": int(comment_subset["video_id"].nunique()),
                "n_active_dates": int(comment_subset["comment_date_local"].nunique()),
                "n_active_weeks": int(comment_subset["week_start_date"].nunique()),
                "dominant_weekday_by_comments": dominant_by_comments,
                "dominant_weekday_by_active_hcc": dominant_by_active_hcc,
                "dominant_daypart": ordered_mode(comment_subset["daypart"], DAYPART_ORDER),
                "peak_hour": peak_hour(comment_subset["hour_local"]),
                "weekend_comment_share": safe_divide(int(comment_subset["is_weekend"].sum()), int(comment_subset["comment_id"].count())),
                "weekday_concentration_score": concentration_score,
                "mean_hcc_weekday_concentration": hcc_weekday_profile.loc[hcc_weekday_profile["hcc_id"].isin(hcc_ids), "weekday_concentration_score"].mean(),
                "median_hcc_weekday_concentration": hcc_weekday_profile.loc[hcc_weekday_profile["hcc_id"].isin(hcc_ids), "weekday_concentration_score"].median(),
                "mean_positive_ratio": hcc_subset["positive_ratio"].mean(),
                "mean_neutral_ratio": hcc_subset["neutral_ratio"].mean(),
                "mean_negative_ratio": hcc_subset["negative_ratio"].mean(),
                "dominant_goal_orientation": ordered_mode(hcc_subset["goal_orientation"], default="Not Observed"),
                "hcc_ids": join_hcc_ids(hcc_ids),
                "interpretation_safe": (
                    f"Pada konteks {brand}, jumlah HCC aktif tertinggi teramati pada hari {dominant_by_active_hcc}. "
                    f"Hasil ini menunjukkan banyaknya komunitas HCC yang muncul pada video berkonteks {brand} pada hari tersebut, "
                    "bukan bukti afiliasi brand, jadwal kerja, atau koordinasi komersial."
                ),
            }
        )
    brand_hcc_temporal_profile = pd.DataFrame(brand_profile_rows)
    write_csv(brand_hcc_temporal_profile, TABLE_DIR / "brand_hcc_temporal_profile.csv")

    brand_weekday_rows = []
    for brand in BRAND_ORDER:
        subset = community_valid[community_valid["brand_label_auto_context"] == brand]
        counts = subset["weekday_name_id"].value_counts().reindex(WEEKDAY_ORDER, fill_value=0)
        total = int(counts.sum())
        row = {
            "brand_label_auto": brand,
            "n_hcc": int(subset["hcc_id"].nunique()),
            "n_community_actor": int(subset["username"].nunique()),
            "n_comments": total,
            "n_unique_videos": int(subset["video_id"].nunique()),
            "dominant_weekday": ordered_mode(subset["weekday_name_id"], WEEKDAY_ORDER),
            "weekend_share": safe_divide(int(subset["is_weekend"].sum()), total),
            "dominant_daypart": ordered_mode(subset["daypart"], DAYPART_ORDER),
            "peak_hour": peak_hour(subset["hour_local"]),
        }
        for day in WEEKDAY_ORDER:
            row[f"comments_{day}"] = int(counts[day])
            row[f"share_{day}"] = safe_divide(int(counts[day]), total)
        brand_weekday_rows.append(row)
    brand_weekday_summary = pd.DataFrame(brand_weekday_rows)
    brand_weekday_summary = brand_weekday_summary[
        ["brand_label_auto", "n_hcc", "n_community_actor", "n_comments", "n_unique_videos"]
        + weekday_count_columns("comments")
        + weekday_count_columns("share")
        + ["dominant_weekday", "weekend_share", "dominant_daypart", "peak_hour"]
    ]
    write_csv(brand_weekday_summary, TABLE_DIR / "brand_weekday_summary.csv")
    brand_lift_rows = []
    overall_share_map = community_weekday_activity_lift.set_index("weekday")["overall_weekday_share"].to_dict()
    for _, row in brand_weekday_summary.iterrows():
        for day in WEEKDAY_ORDER:
            brand_lift_rows.append(
                {
                    "brand_label_auto": row["brand_label_auto"],
                    "weekday_name_id": day,
                    "weekday_number": WEEKDAY_NAME_TO_NUMBER[day],
                    "brand_weekday_share": row[f"share_{day}"],
                    "overall_weekday_share": overall_share_map.get(day, np.nan),
                    "brand_weekday_activity_lift": safe_divide(row[f"share_{day}"], overall_share_map.get(day, np.nan)),
                    "n_comments": row[f"comments_{day}"],
                }
            )
    brand_weekday_activity_lift = pd.DataFrame(brand_lift_rows)
    write_csv(brand_weekday_activity_lift, TABLE_DIR / "brand_weekday_activity_lift.csv")

    video_norm_rows = []

    def append_video_normalized_rows(frame, analysis_level, brand_label="All"):
        grouped = (
            frame.groupby("weekday_number")
            .agg(
                n_comments=("comment_id", "count"),
                n_unique_videos=("video_id", "nunique"),
                n_unique_community_actors=("username", lambda s: s[frame.loc[s.index, "actor_type_primary"] == "Community Actor"].nunique()),
            )
            .reindex(community_weekday_index, fill_value=0)
            .reset_index()
        )
        for _, row in grouped.iterrows():
            video_norm_rows.append(
                {
                    "analysis_level": analysis_level,
                    "brand_label_auto": brand_label,
                    "weekday_number": int(row["weekday_number"]),
                    "weekday_name_id": WEEKDAY_NUMBER_TO_NAME[int(row["weekday_number"])],
                    "n_comments": int(row["n_comments"]),
                    "n_unique_videos": int(row["n_unique_videos"]),
                    "n_unique_community_actors": int(row["n_unique_community_actors"]),
                    "comments_per_active_video": safe_divide(row["n_comments"], row["n_unique_videos"]),
                    "community_accounts_per_active_video": safe_divide(row["n_unique_community_actors"], row["n_unique_videos"]),
                }
            )

    append_video_normalized_rows(valid_comments, "All comments", "All")
    append_video_normalized_rows(community_valid, "Community Actor", "All Community Actor")
    for brand in BRAND_ORDER:
        append_video_normalized_rows(community_valid[community_valid["brand_label_auto_context"] == brand], "Brand context", brand)
    weekday_video_normalized_summary = pd.DataFrame(video_norm_rows)
    write_csv(weekday_video_normalized_summary, TABLE_DIR / "weekday_video_normalized_summary.csv")

    individual_accounts = account_actor_type[account_actor_type["actor_type_primary"].astype(str) == "Individual Actor"].copy()
    individual_valid = valid_comments[valid_comments["actor_type_primary"] == "Individual Actor"].copy()
    individual_all_comments = comments[comments["actor_type_primary"] == "Individual Actor"].copy()
    individual_rows = []
    for _, account_row in individual_accounts.sort_values("username").iterrows():
        username = account_row["username"]
        all_sub = individual_all_comments[individual_all_comments["username"] == username]
        sub = individual_valid[individual_valid["username"] == username]
        weekday_counts = sub["weekday_name_id"].value_counts().reindex(WEEKDAY_ORDER, fill_value=0)
        entropy = normalized_entropy_from_counts(weekday_counts.values)
        concentration_score = np.nan if pd.isna(entropy) else 1 - entropy
        n_active_weeks = int(sub["week_start_date"].nunique())
        n_valid = int(len(sub))
        if n_valid == 0:
            dominant_comment_weekday = "Not Observed"
            dominant_comment_daypart = "Not Observed"
            peak_comment_hour = np.nan
        else:
            dominant_comment_weekday = ordered_mode(sub["weekday_name_id"], WEEKDAY_ORDER)
            dominant_comment_daypart = ordered_mode(sub["daypart"], DAYPART_ORDER)
            peak_comment_hour = peak_hour(sub["hour_local"])
        individual_rows.append(
            {
                "individual_username": clean_identifier(account_row.get("display_name")) or username,
                "individual_subtype": account_row.get("individual_subtype"),
                "n_comments": int(len(all_sub)),
                "n_active_dates": int(sub["comment_date_local"].nunique()),
                "n_active_weeks": n_active_weeks,
                "dominant_comment_weekday": dominant_comment_weekday,
                "dominant_comment_daypart": dominant_comment_daypart,
                "peak_comment_hour": peak_comment_hour,
                "weekday_concentration_score": concentration_score,
                "temporal_profile_reliability": temporal_reliability(n_valid, n_active_weeks),
            }
        )
    individual_actor_comment_weekday_profile = pd.DataFrame(individual_rows)
    write_csv(individual_actor_comment_weekday_profile, TABLE_DIR / "individual_actor_comment_weekday_profile.csv")
    video_metadata["creator"] = video_metadata["creator"].map(clean_identifier)
    upload_text = video_metadata["upload_date"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
    video_metadata["upload_date_parsed"] = pd.to_datetime(upload_text, format="%Y%m%d", errors="coerce")
    video_metadata["upload_weekday_number"] = video_metadata["upload_date_parsed"].dt.dayofweek + 1
    video_metadata["upload_weekday_name_id"] = video_metadata["upload_weekday_number"].map(WEEKDAY_NUMBER_TO_NAME)
    individual_upload_profile = (
        video_metadata.groupby("creator")
        .agg(
            n_uploaded_videos=("video_id", "count"),
            n_valid_upload_dates=("upload_date_parsed", lambda s: s.notna().sum()),
            first_upload_date=("upload_date_parsed", "min"),
            last_upload_date=("upload_date_parsed", "max"),
            dominant_upload_weekday=("upload_weekday_name_id", lambda s: ordered_mode(s, WEEKDAY_ORDER)),
        )
        .reset_index()
        .rename(columns={"creator": "individual_username"})
    )
    write_csv(individual_upload_profile, TABLE_DIR / "individual_actor_upload_weekday_profile.csv")

    def build_interpretation(row: pd.Series) -> str:
        hcc_id = row["hcc_id"]
        if row["temporal_profile_reliability"] == "Insufficient observations" or row["dominant_weekday"] == "Not Observed":
            return (
                f"HCC {hcc_id} memiliki observasi timestamp yang terbatas untuk profil temporal deskriptif. "
                "Pola temporal yang teramati tidak dapat dibaca sebagai bukti jadwal kerja, hubungan komersial, koordinasi terencana, atau status akun sebagai buzzer."
            )
        share_text = format_pct(row["dominant_weekday_share"], decimals=1)
        coverage_text = f"{int(row['dominant_weekday_active_weeks'])} dari {int(row['n_active_weeks'])} minggu aktif" if pd.notna(row["n_active_weeks"]) else "NA minggu aktif"
        return (
            f"HCC {hcc_id} memiliki aktivitas komentar HCC tertinggi pada hari {row['dominant_weekday']} "
            f"dengan proporsi {share_text} dari komentar bertimestamp valid. Aktivitas tersebut teramati pada "
            f"{coverage_text} dan paling sering muncul pada {str(row['dominant_daypart']).lower()}. "
            "Pola ini menunjukkan konsentrasi aktivitas berdasarkan hari yang teramati, bukan bukti jadwal kerja, hubungan komersial, koordinasi terencana, atau status akun sebagai buzzer."
        )

    hcc_temporal_goal_profile = hcc_weekday_profile.copy()
    hcc_temporal_goal_profile["interpretation_safe"] = hcc_temporal_goal_profile.apply(build_interpretation, axis=1)
    hcc_temporal_goal_profile = hcc_temporal_goal_profile[
        [
            "hcc_id",
            "community_size",
            "primary_brand",
            "brand_label_auto",
            "goal_orientation",
            "dominant_sentiment",
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio",
            "narrative_similarity_level",
            "dominant_weekday",
            "dominant_weekday_share",
            "dominant_daypart",
            "peak_hour",
            "weekday_concentration_score",
            "n_active_weeks",
            "dominant_weekday_week_coverage",
            "temporal_profile_reliability",
            "interpretation_safe",
        ]
    ]
    write_csv(hcc_temporal_goal_profile, TABLE_DIR / "hcc_temporal_goal_profile.csv")

    contingency = actor_type_weekday_summary.pivot(index="actor_type_primary", columns="weekday_name_id", values="n_comments").reindex(index=ACTOR_TYPE_ORDER, columns=WEEKDAY_ORDER, fill_value=0)
    chi2_stat, p_value, dof, expected = chi2_contingency(contingency.values)
    n_observations = contingency.values.sum()
    cramers_v = math.sqrt(chi2_stat / (n_observations * (min(contingency.shape) - 1))) if n_observations > 0 else np.nan
    weekday_actor_type_chi_square = pd.DataFrame(
        [
            {
                "test": "Chi-square actor_type_primary x weekday_name_id",
                "chi_square_statistic": chi2_stat,
                "degrees_of_freedom": dof,
                "p_value": p_value,
                "cramers_v": cramers_v,
                "n_observations": int(n_observations),
                "caveat": "Uji chi-square diperlakukan sebagai analisis pendukung karena beberapa komentar berasal dari akun yang sama sehingga observasi comment-level tidak sepenuhnya independen. Ukuran efek Cramer's V dan analisis account-level harus dibaca bersama p-value.",
            }
        ]
    )
    write_csv(weekday_actor_type_chi_square, TABLE_DIR / "weekday_actor_type_chi_square.csv")
    expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
    residual_df = (contingency - expected_df) / np.sqrt(expected_df)
    weekday_actor_type_standardized_residuals = residual_df.reset_index().melt(id_vars="actor_type_primary", var_name="weekday_name_id", value_name="standardized_residual")
    weekday_actor_type_standardized_residuals = (
        weekday_actor_type_standardized_residuals.merge(
            contingency.reset_index().melt(id_vars="actor_type_primary", var_name="weekday_name_id", value_name="observed_count"),
            on=["actor_type_primary", "weekday_name_id"],
            how="left",
        )
        .merge(
            expected_df.reset_index().melt(id_vars="actor_type_primary", var_name="weekday_name_id", value_name="expected_count"),
            on=["actor_type_primary", "weekday_name_id"],
            how="left",
        )
    )
    weekday_actor_type_standardized_residuals["weekday_number"] = weekday_actor_type_standardized_residuals["weekday_name_id"].map(WEEKDAY_NAME_TO_NUMBER)
    weekday_actor_type_standardized_residuals = weekday_actor_type_standardized_residuals.sort_values(["actor_type_primary", "weekday_number"])
    write_csv(weekday_actor_type_standardized_residuals, TABLE_DIR / "weekday_actor_type_standardized_residuals.csv")

    account_rows = []
    account_lookup = account_actor_type.copy()
    account_lookup["comment_username_key"] = account_lookup["display_name"].map(clean_identifier)
    account_lookup["comment_username_key"] = account_lookup["comment_username_key"].combine_first(
        account_lookup["username"].map(clean_identifier)
    )
    account_lookup = account_lookup.set_index("comment_username_key")
    for username, sub in valid_comments.groupby("username"):
        if len(sub) < 3:
            continue
        account_info = account_lookup.loc[username] if username in account_lookup.index else pd.Series(dtype=object)
        actor_type = str(sub["actor_type_primary"].iloc[0])
        temp_row = pd.Series(
            {
                "actor_type_primary": actor_type,
                "username": username,
                "actor_id_anonymous": account_info.get("actor_id_anonymous", np.nan),
                "display_name": account_info.get("display_name", np.nan),
            }
        )
        weekday_counts = sub["weekday_name_id"].value_counts().reindex(WEEKDAY_ORDER, fill_value=0)
        entropy = normalized_entropy_from_counts(weekday_counts.values)
        concentration_score = np.nan if pd.isna(entropy) else 1 - entropy
        dominant_weekday = ordered_mode(sub["weekday_name_id"], WEEKDAY_ORDER)
        n_comments = int(len(sub))
        account_rows.append(
            {
                "public_actor_id": public_actor_identifier(temp_row),
                "actor_type_primary": actor_type,
                "hcc_id": sub["hcc_id"].dropna().iloc[0] if actor_type == "Community Actor" and sub["hcc_id"].notna().any() else np.nan,
                "n_comments": n_comments,
                "n_active_weeks": int(sub["week_start_date"].nunique()),
                "dominant_weekday": dominant_weekday,
                "dominant_weekday_share": safe_divide(int(weekday_counts.get(dominant_weekday, 0)), n_comments) if dominant_weekday in WEEKDAY_ORDER else np.nan,
                "dominant_daypart": ordered_mode(sub["daypart"], DAYPART_ORDER),
                "peak_hour": peak_hour(sub["hour_local"]),
                "weekday_concentration_score": concentration_score,
            }
        )
    account_temporal_profile = pd.DataFrame(account_rows).sort_values(["actor_type_primary", "public_actor_id"])
    write_csv(account_temporal_profile, TABLE_DIR / "account_temporal_profile.csv")
    account_summary_index = pd.MultiIndex.from_product([ACTOR_TYPE_ORDER, WEEKDAY_ORDER], names=["actor_type_primary", "dominant_weekday"])
    account_dominant_weekday_by_actor_type = (
        account_temporal_profile.groupby(["actor_type_primary", "dominant_weekday"])
        .agg(
            n_accounts=("public_actor_id", "count"),
            median_dominant_weekday_share=("dominant_weekday_share", "median"),
            median_weekday_concentration_score=("weekday_concentration_score", "median"),
        )
        .reindex(account_summary_index, fill_value=0)
        .reset_index()
    )
    account_type_totals = account_temporal_profile.groupby("actor_type_primary")["public_actor_id"].count().reindex(ACTOR_TYPE_ORDER, fill_value=0)
    account_dominant_weekday_by_actor_type["account_share_within_actor_type"] = account_dominant_weekday_by_actor_type.apply(lambda row: safe_divide(row["n_accounts"], account_type_totals.get(row["actor_type_primary"], 0)), axis=1)
    account_dominant_weekday_by_actor_type["weekday_number"] = account_dominant_weekday_by_actor_type["dominant_weekday"].map(WEEKDAY_NAME_TO_NUMBER)
    account_dominant_weekday_by_actor_type = account_dominant_weekday_by_actor_type.sort_values(["actor_type_primary", "weekday_number"])
    write_csv(account_dominant_weekday_by_actor_type, TABLE_DIR / "account_dominant_weekday_by_actor_type.csv")

    temporal_method_parameters = pd.DataFrame(
        [
            {"parameter": "TIMESTAMP_SOURCE_TIMEZONE", "value": TIMESTAMP_SOURCE_TIMEZONE, "note": "Only applied to timezone-naive timestamp values."},
            {"parameter": "ANALYSIS_TIMEZONE", "value": ANALYSIS_TIMEZONE, "note": "Main local timezone for weekday and hour features: Jakarta/WIB."},
            {"parameter": "TIMEZONE_SENSITIVITY", "value": "; ".join(TIMEZONE_SENSITIVITY), "note": "WIB versus WITA sensitivity audit."},
            {"parameter": "WEEKDAY_ORDER", "value": "; ".join(WEEKDAY_ORDER), "note": "Explicit mapping, independent from OS locale."},
            {"parameter": "DAYPART_ORDER", "value": "; ".join(DAYPART_ORDER), "note": "Explicit order for daypart summaries."},
            {"parameter": "daypart_boundaries", "value": "Dini Hari 00:00-04:59; Pagi 05:00-10:59; Siang 11:00-14:59; Sore 15:00-17:59; Malam 18:00-23:59", "note": "Hour ranges are inclusive."},
            {"parameter": "minimum_comments_temporal_profile", "value": MIN_COMMENTS_TEMPORAL_PROFILE, "note": "Minimum valid timestamp comments for descriptive HCC/account temporal profile."},
            {"parameter": "minimum_active_weeks", "value": MIN_ACTIVE_WEEKS, "note": "Minimum active weeks for descriptively sufficient recurrence."},
            {"parameter": "entropy_formula", "value": "-sum(p_i * log(p_i)) / log(7)", "note": "Normalized weekday entropy."},
            {"parameter": "concentration_formula", "value": "1 - weekday_entropy_normalized", "note": "Higher means more concentrated weekday activity."},
            {"parameter": "weekday_lift_formula", "value": "community_weekday_share / overall_weekday_share", "note": "Relative Community Actor weekday activity compared with all comments."},
            {"parameter": "weekend_definition", "value": "Sabtu=True; Minggu=True; other weekdays=False", "note": "Indonesian weekday labels."},
            {"parameter": "timestamp_filtering_rules", "value": "Invalid timestamps excluded from temporal summaries; future/outside flags audited but not filtered by default.", "note": f"TEMPORAL_MIN_DATE={TEMPORAL_MIN_DATE}; TEMPORAL_MAX_DATE={TEMPORAL_MAX_DATE}"},
            {"parameter": "exclusive_hcc_brand_classification", "value": "brand_label_auto from output/tables/hcc_brand_profile_auto.csv", "note": "Each HCC belongs to exactly one RM1 auto-labeled category."},
            {"parameter": "multilabel_hcc_brand_incidence", "value": "brand_combo from output/tables/hcc_brand_profile_auto.csv", "note": "Uses RM1 active-brand representation from video metadata hashtag scores; mixed HCC can be counted once per active brand."},
            {"parameter": "rm1_brand_activation_source", "value": "RM1 Section 16.1.3: active_brands are brands with hashtag metadata frequency score > 0; brand_combo stores those active brands; brand_label_auto uses top_brand_share >= 0.60 for dominant single-brand labels.", "note": f"Dominant share threshold documented from RM1: {DOMINANT_SHARE_THRESHOLD_RM1:.2f}. No new activation threshold is introduced here."},
            {"parameter": "hcc_brand_context_note", "value": "Pengelompokan brand pada HCC menunjukkan konteks video yang dikomentari oleh anggota komunitas berdasarkan hashtag metadata video. Label tersebut tidak berarti seluruh anggota HCC menulis hashtag, mendukung brand, berafiliasi dengan brand, atau membahas brand secara eksplisit dalam setiap komentar.", "note": "Brand context interpretation boundary."},
            {"parameter": "chi_square_caveat", "value": "Uji chi-square diperlakukan sebagai analisis pendukung karena beberapa komentar berasal dari akun yang sama sehingga observasi comment-level tidak sepenuhnya independen. Ukuran efek Cramer's V dan analisis account-level harus dibaca bersama p-value.", "note": "Do not infer substantive difference from p-value alone."},
            {"parameter": "methodological_note", "value": "Analisis hari dan jam aktivitas hanya menggambarkan distribusi waktu komentar yang teramati. Konsentrasi aktivitas pada hari atau jam tertentu tidak membuktikan adanya jadwal kerja, hubungan komersial, koordinasi terencana, maupun status akun sebagai buzzer.", "note": "Interpretation boundary."},
        ]
    )
    write_csv(temporal_method_parameters, TABLE_DIR / "temporal_method_parameters.csv")

    nodes_source = first_existing([INPUT_PATHS["gephi_hcc_nodes_actor_type"], INPUT_PATHS["gephi_hcc_nodes_sentiment"], INPUT_PATHS["gephi_hcc_nodes"]])
    edges_source = first_existing([INPUT_PATHS["gephi_hcc_edges_actor_type"], INPUT_PATHS["gephi_hcc_edges_sentiment"], INPUT_PATHS["gephi_hcc_edges"]])
    gephi_nodes_temporal = read_csv_safe(nodes_source)
    normalize_identifier_column(gephi_nodes_temporal, ["id", "label", "community"])
    node_hcc_key = gephi_nodes_temporal["community"].map(clean_identifier)
    node_attrs = hcc_weekday_profile[
        [
            "hcc_id",
            "dominant_weekday",
            "dominant_weekday_share",
            "dominant_daypart",
            "peak_hour",
            "weekday_entropy_normalized",
            "weekday_concentration_score",
            "n_active_weeks",
            "dominant_weekday_week_coverage",
            "temporal_profile_reliability",
        ]
    ].rename(
        columns={
            "dominant_weekday": "hcc_dominant_weekday",
            "dominant_weekday_share": "hcc_dominant_weekday_share",
            "dominant_daypart": "hcc_dominant_daypart",
            "peak_hour": "hcc_peak_hour",
            "weekday_entropy_normalized": "hcc_weekday_entropy_normalized",
            "weekday_concentration_score": "hcc_weekday_concentration_score",
            "n_active_weeks": "hcc_n_active_weeks",
            "dominant_weekday_week_coverage": "hcc_dominant_weekday_week_coverage",
            "temporal_profile_reliability": "hcc_temporal_profile_reliability",
        }
    )
    gephi_nodes_temporal["_hcc_id_for_merge"] = node_hcc_key
    gephi_nodes_temporal = gephi_nodes_temporal.merge(node_attrs, left_on="_hcc_id_for_merge", right_on="hcc_id", how="left")
    gephi_nodes_temporal = gephi_nodes_temporal.drop(columns=["_hcc_id_for_merge", "hcc_id"])
    write_csv(gephi_nodes_temporal, GEPHI_DIR / "gephi_hcc_nodes_temporal.csv")
    edge_dest = GEPHI_DIR / "gephi_hcc_edges_temporal.csv"
    shutil.copyfile(edges_source, edge_dest)
    edge_hash_validation = pd.DataFrame(
        [
            {"metric": "edge_source_path", "value": str(edges_source.relative_to(BASE_DIR))},
            {"metric": "edge_output_path", "value": str(edge_dest.relative_to(BASE_DIR))},
            {"metric": "edge_source_sha256", "value": sha256_file(edges_source)},
            {"metric": "edge_output_sha256", "value": sha256_file(edge_dest)},
            {"metric": "edge_hash_equal", "value": sha256_file(edges_source) == sha256_file(edge_dest)},
        ]
    )
    write_csv(edge_hash_validation, TABLE_DIR / "gephi_edge_hash_validation.csv")
    assert edge_hash_validation.loc[edge_hash_validation["metric"] == "edge_hash_equal", "value"].iloc[0] == True

    # Visualizations
    plot_df = community_weekday_summary.copy()
    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.barplot(data=plot_df, x="weekday_name_id", y="n_comments", order=WEEKDAY_ORDER, color="#4C78A8", ax=ax)
    ax.set_title(f"Community Actor comments by weekday (n={community_total_comments:,})")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Number of comments")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", padding=2, fontsize=8)
    save_figure(fig, "community_actor_comments_by_weekday_absolute.png")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.barplot(data=plot_df, x="weekday_name_id", y="comment_share", order=WEEKDAY_ORDER, color="#59A14F", ax=ax)
    ax.set_title(f"Community Actor comments by weekday share (n={community_total_comments:,})")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Share of Community Actor comments")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    for patch, value in zip(ax.patches, plot_df["comment_share"]):
        ax.text(patch.get_x() + patch.get_width() / 2, patch.get_height(), format_pct(value), ha="center", va="bottom", fontsize=8)
    save_figure(fig, "community_actor_comments_by_weekday_percentage.png")

    pivot_share = actor_type_weekday_summary.pivot(index="actor_type_primary", columns="weekday_name_id", values="comment_share_within_actor_type").reindex(index=ACTOR_TYPE_ORDER, columns=WEEKDAY_ORDER)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    pivot_share.plot(kind="barh", stacked=True, ax=ax, colormap="tab20c")
    ax.set_title(f"Weekday distribution by actor type (100 pct, n={len(valid_comments):,})")
    ax.set_xlabel("Share within actor type")
    ax.set_ylabel("Actor type")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.legend(title="Weekday", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "actor_type_weekday_distribution_100pct.png")

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=actor_type_weekday_summary, x="weekday_name_id", y="n_comments", hue="actor_type_primary", order=WEEKDAY_ORDER, hue_order=ACTOR_TYPE_ORDER, ax=ax)
    ax.set_yscale("log")
    ax.set_title(f"Comments by actor type and weekday (log scale, n={len(valid_comments):,})")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Number of comments (log scale)")
    ax.legend(title="Actor type")
    save_figure(fig, "actor_type_weekday_absolute.png")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.lineplot(data=community_weekday_activity_lift, x="weekday", y="weekday_activity_lift", marker="o", sort=False, ax=ax, color="#E15759")
    ax.axhline(1, color="black", linewidth=1, linestyle="--")
    ax.set_title(f"Community Actor weekday activity lift (n={community_total_comments:,})")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Lift versus all comments")
    save_figure(fig, "community_weekday_activity_lift.png")

    heat = community_weekday_hour_matrix.pivot(index="weekday_name_id", columns="hour_local", values="n_comments").reindex(index=WEEKDAY_ORDER, columns=range(24), fill_value=0)
    fig, ax = plt.subplots(figsize=(13, 4.8))
    sns.heatmap(heat, cmap="YlGnBu", ax=ax, cbar_kws={"label": "Comments"})
    ax.set_title(f"Community Actor weekday x hour heatmap (WIB, n={community_total_comments:,})")
    ax.set_xlabel("Hour local (WIB)")
    ax.set_ylabel("Weekday")
    save_figure(fig, "community_weekday_hour_heatmap.png")

    daypart_heat = community_valid.groupby(["weekday_name_id", "daypart"], observed=False).size().unstack("daypart", fill_value=0).reindex(index=WEEKDAY_ORDER, columns=DAYPART_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.heatmap(daypart_heat, cmap="BuPu", annot=True, fmt="d", ax=ax, cbar_kws={"label": "Comments"})
    ax.set_title(f"Community Actor weekday x daypart heatmap (n={community_total_comments:,})")
    ax.set_xlabel("Daypart (WIB)")
    ax.set_ylabel("Weekday")
    save_figure(fig, "community_weekday_daypart_heatmap.png")

    hcc_dom_counts = hcc_weekday_profile["dominant_weekday"].value_counts().reindex(WEEKDAY_ORDER + ["Not Observed"], fill_value=0).reset_index()
    hcc_dom_counts.columns = ["dominant_weekday", "n_hcc"]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.barplot(data=hcc_dom_counts, x="dominant_weekday", y="n_hcc", color="#F28E2B", ax=ax)
    ax.set_title(f"HCC dominant weekday distribution (n={len(hcc_weekday_profile):,} HCC)")
    ax.set_xlabel("Dominant weekday (WIB)")
    ax.set_ylabel("Number of HCC")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", padding=2, fontsize=8)
    save_figure(fig, "hcc_dominant_weekday_distribution.png")

    concentration_plot = hcc_weekday_profile.sort_values("weekday_concentration_score", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(11, 5.5))
    palette = {"Descriptively sufficient": "#4C78A8", "Limited recurrence": "#F28E2B", "Insufficient observations": "#B0B0B0"}
    sns.barplot(data=concentration_plot, x="hcc_id", y="weekday_concentration_score", hue="temporal_profile_reliability", dodge=False, palette=palette, ax=ax)
    ax.set_title("HCC weekday concentration score; grey/orange indicate limited or insufficient profiles")
    ax.set_xlabel("HCC ID")
    ax.set_ylabel("Weekday concentration score")
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.legend(title="Reliability", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "hcc_weekday_concentration.png")

    top_hcc = hcc_weekday_profile.sort_values("n_valid_timestamp_comments", ascending=False).head(15)
    top_share = top_hcc.set_index("hcc_id")[weekday_count_columns("share")]
    top_share.columns = WEEKDAY_ORDER
    fig, ax = plt.subplots(figsize=(10, 6))
    top_share.plot(kind="barh", stacked=True, ax=ax, colormap="tab20c")
    ax.set_title("Top HCC weekday profiles by valid timestamp comments (max 15 HCC)")
    ax.set_xlabel("Share within HCC")
    ax.set_ylabel("HCC ID")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.legend(title="Weekday", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "top_hcc_weekday_profiles.png")

    top_ids = top_hcc["hcc_id"].tolist()
    combined_columns = [f"{day[:3]} {hour:02d}" for day in WEEKDAY_ORDER for hour in range(24)]
    rows = []
    for hcc_id in top_ids:
        sub = hcc_weekday_hour_matrix[hcc_weekday_hour_matrix["hcc_id"] == hcc_id]
        pivot = sub.pivot(index="weekday_name_id", columns="hour_local", values="n_comments").reindex(index=WEEKDAY_ORDER, columns=range(24), fill_value=0)
        row_values = []
        for day in WEEKDAY_ORDER:
            row_values.extend(pivot.loc[day].tolist())
        rows.append(row_values)
    top_hcc_hour_matrix = pd.DataFrame(rows, index=top_ids, columns=combined_columns)
    fig, ax = plt.subplots(figsize=(15, 5.8))
    sns.heatmap(top_hcc_hour_matrix, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Comments"})
    ax.set_title("Top HCC weekday x hour heatmap (WIB)")
    ax.set_xlabel("Weekday-hour")
    ax.set_ylabel("HCC ID")
    ax.set_xticks([i for i in range(0, len(combined_columns), 12)])
    ax.set_xticklabels([combined_columns[i] for i in range(0, len(combined_columns), 12)], rotation=45, ha="right", fontsize=7)
    save_figure(fig, "top_hcc_weekday_hour_heatmap.png")

    brand_share = brand_weekday_summary.set_index("brand_label_auto")[weekday_count_columns("share")]
    brand_share.columns = WEEKDAY_ORDER
    fig, ax = plt.subplots(figsize=(10, 5.5))
    brand_share.plot(kind="barh", stacked=True, ax=ax, colormap="tab20c")
    ax.set_title(f"Brand-context weekday distribution for Community Actor comments (n={community_total_comments:,})")
    ax.set_xlabel("Share within brand context")
    ax.set_ylabel("Brand context")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.legend(title="Weekday", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "brand_weekday_distribution_100pct.png")

    brand_abs = brand_weekday_summary.set_index("brand_label_auto")[weekday_count_columns("comments")]
    brand_abs.columns = WEEKDAY_ORDER
    fig, ax = plt.subplots(figsize=(10, 5.5))
    brand_abs.plot(kind="barh", stacked=True, ax=ax, colormap="tab20c")
    ax.set_title("Brand-context weekday activity for Community Actor comments")
    ax.set_xlabel("Number of comments")
    ax.set_ylabel("Brand context")
    ax.legend(title="Weekday", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "brand_weekday_activity_absolute.png")

    weekend_share = (
        valid_comments.groupby("actor_type_primary")["is_weekend"]
        .agg(lambda series: safe_divide(int(series.sum()), len(series)))
        .reindex(ACTOR_TYPE_ORDER)
        .reset_index()
    )
    weekend_share.columns = ["actor_type_primary", "weekend_share"]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.barplot(data=weekend_share, x="actor_type_primary", y="weekend_share", order=ACTOR_TYPE_ORDER, color="#76B7B2", ax=ax)
    ax.set_title("Weekend comment share by actor type")
    ax.set_xlabel("Actor type")
    ax.set_ylabel("Weekend share")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    for patch, value in zip(ax.patches, weekend_share["weekend_share"]):
        ax.text(patch.get_x() + patch.get_width() / 2, patch.get_height(), format_pct(value), ha="center", va="bottom", fontsize=8)
    save_figure(fig, "weekend_share_by_actor_type.png")

    actor_daypart_pivot = actor_type_daypart_summary.pivot(index="actor_type_primary", columns="daypart", values="comment_share_within_actor_type").reindex(index=ACTOR_TYPE_ORDER, columns=DAYPART_ORDER)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    actor_daypart_pivot.plot(kind="barh", stacked=True, ax=ax, colormap="Set2")
    ax.set_title("Daypart distribution by actor type")
    ax.set_xlabel("Share within actor type")
    ax.set_ylabel("Actor type")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.legend(title="Daypart", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "dominant_daypart_by_actor_type.png")

    recurrence_plot = hcc_weekday_profile.sort_values("dominant_weekday_week_coverage", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(11, 5.5))
    sns.barplot(data=recurrence_plot, x="hcc_id", y="dominant_weekday_week_coverage", hue="temporal_profile_reliability", dodge=False, palette=palette, ax=ax)
    ax.set_title("Dominant weekday recurrence coverage by HCC")
    ax.set_xlabel("HCC ID")
    ax.set_ylabel("Coverage of dominant weekday across active weeks")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.legend(title="Reliability", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "weekday_recurrence_by_hcc.png")

    brand_lift_heat = brand_weekday_activity_lift.pivot(index="brand_label_auto", columns="weekday_name_id", values="brand_weekday_activity_lift").reindex(index=BRAND_ORDER, columns=WEEKDAY_ORDER)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    sns.heatmap(brand_lift_heat, cmap="vlag", center=1, annot=True, fmt=".2f", ax=ax, cbar_kws={"label": "Lift vs all comments"})
    ax.set_title("Weekday activity lift by brand context")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Brand context")
    save_figure(fig, "weekday_activity_lift_by_brand.png")

    # Additional HCC brand context visualizations.
    exclusive_plot = hcc_brand_exclusive_summary[hcc_brand_exclusive_summary["brand_label_auto"] != "Total"].copy()
    fig, ax = plt.subplots(figsize=(10, 4.8))
    sns.barplot(data=exclusive_plot, x="brand_label_auto", y="n_hcc", order=BRAND_ORDER, color="#4C78A8", ax=ax)
    ax.set_title(f"HCC count by exclusive brand label (total HCC={total_valid_hcc})")
    ax.set_xlabel("Exclusive brand label from RM1")
    ax.set_ylabel("Number of HCC")
    ax.tick_params(axis="x", rotation=30)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", padding=2, fontsize=8)
    save_figure(fig, "hcc_count_by_exclusive_brand_label.png")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.barplot(data=hcc_brand_multilabel_summary, x="brand", y="n_hcc_associated", order=INDIVIDUAL_BRANDS, color="#59A14F", ax=ax)
    ax.set_title(f"Multi-label HCC incidence by brand; summed bars may exceed {total_valid_hcc} HCC")
    ax.set_xlabel("Brand context from RM1 brand_combo")
    ax.set_ylabel("Number of associated HCC")
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", padding=2, fontsize=8)
    save_figure(fig, "hcc_multilabel_incidence_by_brand.png")

    contribution = hcc_brand_multilabel_summary.set_index("brand")[
        ["n_hcc_single_brand", "n_hcc_from_mixed_2", "n_hcc_from_mixed_3plus"]
    ].reindex(INDIVIDUAL_BRANDS)
    contribution.columns = ["Single-brand HCC", "Mixed_2 contribution", "Mixed_3plus contribution"]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    contribution.plot(kind="bar", stacked=True, ax=ax, color=["#4C78A8", "#F28E2B", "#E15759"])
    ax.set_title("Single and mixed HCC contribution by brand context")
    ax.set_xlabel("Brand context")
    ax.set_ylabel("Number of associated HCC")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="HCC source")
    for container in ax.containers:
        labels = [f"{int(patch.get_height())}" if patch.get_height() > 0 else "" for patch in container]
        ax.bar_label(container, labels=labels, label_type="center", fontsize=8)
    save_figure(fig, "hcc_single_vs_cross_brand_contribution.png")

    active_heat = brand_weekday_active_hcc_summary.pivot(index="brand", columns="weekday_name_id", values="n_active_hcc").reindex(index=INDIVIDUAL_BRANDS, columns=WEEKDAY_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.heatmap(active_heat, cmap="YlGnBu", annot=True, fmt="d", ax=ax, cbar_kws={"label": "Active HCC"})
    ax.set_title("Active HCC count by brand context and weekday")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Brand context")
    save_figure(fig, "brand_weekday_active_hcc_heatmap.png")

    fig, ax = plt.subplots(figsize=(9, 4.8))
    sns.lineplot(
        data=brand_weekday_active_hcc_summary,
        x="weekday_name_id",
        y="n_active_hcc",
        hue="brand",
        hue_order=INDIVIDUAL_BRANDS,
        marker="o",
        sort=False,
        ax=ax,
    )
    ax.set_title("Active HCC count by brand context and weekday")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Number of active HCC")
    ax.legend(title="Brand context", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_figure(fig, "brand_weekday_active_hcc_lines.png")

    brand_comment_heat = brand_weekday_active_hcc_summary.pivot(index="brand", columns="weekday_name_id", values="n_comments").reindex(index=INDIVIDUAL_BRANDS, columns=WEEKDAY_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    sns.heatmap(brand_comment_heat, cmap="YlOrRd", annot=True, fmt="d", ax=ax, cbar_kws={"label": "Community Actor comments"})
    ax.set_title("Community Actor comment count by brand context and weekday")
    ax.set_xlabel("Weekday (WIB)")
    ax.set_ylabel("Brand context")
    save_figure(fig, "brand_weekday_comments_heatmap.png")

    fig, ax = plt.subplots(figsize=(7, 4.8))
    sns.barplot(data=hcc_brand_scope_summary, x="brand_scope", y="n_hcc", color="#76B7B2", ax=ax)
    ax.set_title("Single-brand, cross-brand, and unidentified HCC counts")
    ax.set_xlabel("HCC brand scope")
    ax.set_ylabel("Number of HCC")
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", padding=2, fontsize=8)
    save_figure(fig, "single_vs_cross_brand_hcc_summary.png")

    primary_vs_associated = hcc_brand_multilabel_summary[["brand", "n_hcc_as_primary_brand", "n_hcc_associated"]].melt(
        id_vars="brand", var_name="count_type", value_name="n_hcc"
    )
    primary_vs_associated["count_type"] = primary_vs_associated["count_type"].map(
        {"n_hcc_as_primary_brand": "Primary brand HCC", "n_hcc_associated": "Associated HCC"}
    )
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    sns.barplot(data=primary_vs_associated, x="brand", y="n_hcc", hue="count_type", order=INDIVIDUAL_BRANDS, ax=ax)
    ax.set_title("Primary versus multi-label associated HCC by brand context")
    ax.set_xlabel("Brand context")
    ax.set_ylabel("Number of HCC")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="Count type")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", padding=2, fontsize=8)
    save_figure(fig, "brand_hcc_primary_vs_associated.png")

    input_checksums_after = {str(path): sha256_file(path) for path in [Path(p) for p in input_checksums_before.keys()]}
    checksum_unchanged = all(input_checksums_before[path] == input_checksums_after[path] for path in input_checksums_before)
    output_tables = sorted([path.name for path in TABLE_DIR.glob("*") if path.is_file()])
    if "temporal_final_validation_report.csv" not in output_tables:
        output_tables = sorted(output_tables + ["temporal_final_validation_report.csv"])
    output_visualizations = sorted([path.name for path in VIS_DIR.glob("*.png")])
    output_gephi = sorted([path.name for path in GEPHI_DIR.glob("*.csv")])
    dominant_weekday_actor_type = (
        actor_type_weekday_summary.sort_values(["actor_type_primary", "n_comments"], ascending=[True, False])
        .groupby("actor_type_primary", as_index=False)
        .first()[["actor_type_primary", "weekday_name_id", "n_comments"]]
    )
    community_count_peak = community_weekday_summary.sort_values("n_comments", ascending=False).iloc[0]
    community_lift_peak = community_weekday_activity_lift.sort_values("weekday_activity_lift", ascending=False).iloc[0]
    community_daypart_peak = ordered_mode(community_valid["daypart"], DAYPART_ORDER)
    community_peak_hour = peak_hour(community_valid["hour_local"])
    hcc_sufficient_count = int((hcc_weekday_profile["temporal_profile_reliability"] == "Descriptively sufficient").sum())
    hcc_dominant_weekday_distribution = hcc_weekday_profile["dominant_weekday"].value_counts().reindex(WEEKDAY_ORDER + ["Not Observed"], fill_value=0)
    hcc_concentration_distribution = hcc_weekday_profile["weekday_concentration_level"].value_counts()
    hcc_highest_concentration = hcc_weekday_profile.sort_values("weekday_concentration_score", ascending=False).head(1)
    hcc_highest_recurrence = hcc_weekday_profile.sort_values(["dominant_weekday_week_coverage", "n_active_weeks"], ascending=False).head(1)
    hcc_exclusive_counts = hcc_brand_exclusive_summary.loc[
        hcc_brand_exclusive_summary["brand_label_auto"] != "Total",
        ["brand_label_auto", "n_hcc"],
    ]
    scope_counts = hcc_brand_scope_summary[["brand_scope", "n_hcc"]]
    multilabel_associated_counts = hcc_brand_multilabel_summary[
        [
            "brand",
            "n_hcc_associated",
            "n_hcc_as_primary_brand",
            "n_hcc_single_brand",
            "n_hcc_from_mixed_2",
            "n_hcc_from_mixed_3plus",
        ]
    ]
    top_active_hcc_day_by_brand = (
        brand_weekday_active_hcc_summary.sort_values(["brand", "n_active_hcc", "weekday_number"], ascending=[True, False, True])
        .groupby("brand", as_index=False)
        .first()[["brand", "weekday_name_id", "n_active_hcc"]]
    )
    top_comment_day_by_brand = (
        brand_weekday_active_hcc_summary.sort_values(["brand", "n_comments", "weekday_number"], ascending=[True, False, True])
        .groupby("brand", as_index=False)
        .first()[["brand", "weekday_name_id", "n_comments"]]
    )
    dominant_weekday_comparison_by_brand = brand_hcc_temporal_profile[
        ["brand", "dominant_weekday_by_comments", "dominant_weekday_by_active_hcc"]
    ].copy()
    dominant_weekday_comparison_by_brand["same_dominant_weekday"] = (
        dominant_weekday_comparison_by_brand["dominant_weekday_by_comments"]
        == dominant_weekday_comparison_by_brand["dominant_weekday_by_active_hcc"]
    )
    final_report_rows = [
        {"metric": "commit_head_before_implementation", "value": PRE_IMPLEMENTATION_HEAD},
        {"metric": "pre_existing_modified_files", "value": "; ".join(PRE_IMPLEMENTATION_MODIFIED_FILES)},
        {"metric": "valid_timestamp_coverage", "value": safe_divide(int(comments["timestamp_utc"].notna().sum()), len(comments))},
        {"metric": "timestamp_range_utc", "value": f"{comments['timestamp_utc'].min()} to {comments['timestamp_utc'].max()}"},
        {"metric": "timestamp_range_local_wib", "value": f"{comments['timestamp_local'].min()} to {comments['timestamp_local'].max()}"},
        {"metric": "analysis_timezone", "value": ANALYSIS_TIMEZONE},
        {"metric": "wib_wita_changed_weekday_percentage", "value": weekday_changed_percentage_wib_vs_wita},
        {"metric": "community_actor_highest_comment_weekday", "value": f"{community_count_peak['weekday_name_id']} ({int(community_count_peak['n_comments'])} comments)"},
        {"metric": "community_actor_highest_lift_weekday", "value": f"{community_lift_peak['weekday']} (lift={community_lift_peak['weekday_activity_lift']:.3f})"},
        {"metric": "community_actor_dominant_daypart", "value": community_daypart_peak},
        {"metric": "community_actor_peak_hour", "value": community_peak_hour},
        {"metric": "hcc_descriptively_sufficient_profiles", "value": hcc_sufficient_count},
        {"metric": "hcc_with_highest_concentration", "value": hcc_highest_concentration[["hcc_id", "weekday_concentration_score"]].to_dict("records")},
        {"metric": "hcc_with_highest_recurrence", "value": hcc_highest_recurrence[["hcc_id", "dominant_weekday_week_coverage", "n_active_weeks"]].to_dict("records")},
        {"metric": "chi_square_and_cramers_v", "value": weekday_actor_type_chi_square[["chi_square_statistic", "degrees_of_freedom", "p_value", "cramers_v"]].to_dict("records")},
        {"metric": "exclusive_hcc_count_by_brand_label", "value": hcc_exclusive_counts.to_dict("records")},
        {"metric": "hcc_brand_scope_counts", "value": scope_counts.to_dict("records")},
        {"metric": "multilabel_hcc_associated_counts", "value": multilabel_associated_counts.to_dict("records")},
        {"metric": "top_active_hcc_day_by_brand", "value": top_active_hcc_day_by_brand.to_dict("records")},
        {"metric": "top_comment_day_by_brand", "value": top_comment_day_by_brand.to_dict("records")},
        {"metric": "dominant_weekday_comments_vs_active_hcc_by_brand", "value": dominant_weekday_comparison_by_brand.to_dict("records")},
        {"metric": "input_checksums_unchanged", "value": checksum_unchanged},
        {"metric": "notebook_run_end_to_end", "value": True},
        {"metric": "tables_created", "value": "; ".join(output_tables)},
        {"metric": "visualizations_created", "value": "; ".join(output_visualizations)},
        {"metric": "gephi_files_created", "value": "; ".join(output_gephi)},
    ]
    final_validation_report = pd.DataFrame(final_report_rows)
    write_csv(final_validation_report, TABLE_DIR / "temporal_final_validation_report.csv")

    print("FINAL VALIDATION REPORT")
    print("Valid timestamp coverage:", format_pct(final_report_rows[2]["value"], decimals=2))
    print("UTC range:", final_report_rows[3]["value"])
    print("WIB range:", final_report_rows[4]["value"])
    print("WIB-WITA changed weekday percentage:", format_pct(weekday_changed_percentage_wib_vs_wita, decimals=2))
    print("\nComments per actor type:")
    print(comments["actor_type_primary"].value_counts().reindex(ACTOR_TYPE_ORDER, fill_value=0).to_string())
    print("\nDominant weekday per actor type:")
    print(dominant_weekday_actor_type.to_string(index=False))
    print("\nCommunity Actor weekday lift:")
    print(community_weekday_activity_lift[["weekday", "community_comment_count", "weekday_activity_lift"]].to_string(index=False))
    print("\nWeekend share per actor type:")
    print(weekend_share.to_string(index=False))
    print("\nHCC dominant weekday distribution:")
    print(hcc_dominant_weekday_distribution.to_string())
    print("\nHCC concentration distribution:")
    print(hcc_concentration_distribution.to_string())
    print("\nBrand x weekday summary:")
    print(brand_weekday_summary[["brand_label_auto", "n_comments", "dominant_weekday", "weekend_share", "dominant_daypart", "peak_hour"]].to_string(index=False))
    print("\nExclusive HCC count by brand label:")
    print(hcc_exclusive_counts.to_string(index=False))
    print("\nSingle-brand / cross-brand / unidentified HCC counts:")
    print(scope_counts.to_string(index=False))
    print("\nMulti-label HCC incidence and mixed contribution by brand:")
    print(multilabel_associated_counts.to_string(index=False))
    print("\nDay with highest active HCC count by brand:")
    print(top_active_hcc_day_by_brand.to_string(index=False))
    print("\nDay with highest Community Actor comment count by brand:")
    print(top_comment_day_by_brand.to_string(index=False))
    print("\nDominant weekday by comments versus active HCC:")
    print(dominant_weekday_comparison_by_brand.to_string(index=False))
    print("\nChi-square and Cramer's V:")
    print(weekday_actor_type_chi_square.to_string(index=False))
    print("\nAll input checksums unchanged:", checksum_unchanged)
    print("Tables:", len(output_tables), "Visualizations:", len(output_visualizations), "Gephi files:", len(output_gephi))
    return {
        "comments": comments,
        "valid_comments": valid_comments,
        "community_valid": community_valid,
        "actor_type_weekday_summary": actor_type_weekday_summary,
        "community_weekday_activity_lift": community_weekday_activity_lift,
        "hcc_weekday_profile": hcc_weekday_profile,
        "brand_weekday_summary": brand_weekday_summary,
        "hcc_brand_exclusive_summary": hcc_brand_exclusive_summary,
        "hcc_brand_multilabel_summary": hcc_brand_multilabel_summary,
        "hcc_brand_membership_long": hcc_brand_membership_long,
        "brand_weekday_active_hcc_summary": brand_weekday_active_hcc_summary,
        "brand_hcc_temporal_profile": brand_hcc_temporal_profile,
        "weekday_actor_type_chi_square": weekday_actor_type_chi_square,
        "final_validation_report": final_validation_report,
    }

## 5. Audit Input dan Checksum

Pipeline mencatat HEAD awal, file yang sudah modified sebelum pekerjaan, checksum seluruh input penting, shape, dan daftar kolom input.

## 6. Load Data

Input utama: dataset komentar, metadata video, node/edge HCC, comment_sentiment, hcc_sentiment_goals_summary, account_actor_type, individual_hcc_association, dan hcc_brand_profile_auto.

## 7. Validasi Actor Type dan HCC

Setiap komentar memperoleh tepat satu actor type. Community Actor dipetakan langsung dari output/gephi/gephi_hcc_nodes.csv; associated_hcc_ids dan primary_associated_hcc tidak dipakai untuk menentukan keanggotaan HCC dalam analisis temporal.

## 8. Timestamp Audit

Notebook menyimpan contoh timestamp mentah, menghitung valid/invalid timestamp, timezone-aware, timezone-naive, dan status offset eksplisit.

## 9. Timezone Conversion

Fitur weekday dan hour dihitung setelah timestamp dikonversi ke Asia/Jakarta / WIB. Sensitivitas Asia/Jakarta versus Asia/Makassar tetap disimpan sebagai audit.

## 10. Timezone Sensitivity

Notebook menghitung n_comments_weekday_changed_wib_vs_wita dan weekday_changed_percentage_wib_vs_wita serta menyimpannya ke timezone_sensitivity_audit.csv.

## 11. Temporal Feature Engineering

Fitur temporal yang dibuat: comment_date_local, comment_year, comment_month, comment_year_month, iso_week, week_start_date, weekday_number, weekday_name_id, hour_local, minute_local, is_weekend, dan daypart. Mapping hari dan daypart eksplisit, tidak bergantung pada locale sistem operasi.

## 12. Data Quality Audit

Audit mencakup coverage timestamp, timestamp masa depan, duplicate comment_id, comment tanpa actor type, comment tanpa video_id, Community Actor tanpa HCC valid, dan flag timestamp_quality_status.

## 13. Actor-Type Weekday Summary

Ringkasan comment-level per actor type dan weekday menghitung raw count, normalized share, unique actors, unique videos, active HCC, video/account normalization, dan rasio sentimen.

## 14. Community Actor Weekday Summary

Tabel khusus Community Actor menghitung share komentar, akun anggota HCC aktif, active HCC, active video, rasio sentimen, dominant sentiment, dan konteks goal orientation.

## 15. Baseline dan Weekday Activity Lift

Weekday lift membandingkan share komentar Community Actor pada hari tertentu dengan baseline seluruh komentar pada hari yang sama. Lift tinggi tidak membuktikan jadwal koordinasi.

## 16. Account-Level Temporal Profile

Profil account-level dibuat untuk akun dengan minimal tiga komentar bertimestamp valid agar akun yang sangat aktif tidak sepenuhnya mendominasi kesimpulan comment-level. Mass Actor ditampilkan memakai actor_id_anonymous.

## 17. HCC-Level Weekday Profile

Profil HCC menghitung dominant weekday, entropy, concentration score, weekend share, daypart, peak hour, active weeks, recurrence coverage, reliability, dan label konsentrasi Low/Moderate/High atau Insufficient.

## 18. Week-Level Recurrence

Recurrence mingguan membedakan hari dominan yang muncul berulang dari pola yang hanya berasal dari satu periode singkat. Pola tidak disebut rutin jika active-week recurrence terbatas.

## 19. Weekday x Hour Analysis

Matriks weekday x hour dibuat untuk actor type, Community Actor, dan HCC dengan observasi timestamp valid yang cukup.

## 20. Daypart Analysis

Daypart dihitung untuk actor type, HCC, dan konteks brand dengan urutan Dini Hari, Pagi, Siang, Sore, Malam.

## 21. Brand x Weekday Analysis

Brand menunjukkan konteks video yang dikomentari, bukan kepemilikan atau afiliasi akun. Ringkasan brand_label_auto menghitung komentar Community Actor per konteks brand dan hari.

## 21b. Jumlah Komunitas HCC Per Konteks Brand

Notebook membedakan dua pendekatan: Exclusive HCC Brand Classification berbasis brand_label_auto, dan Multi-label HCC-Brand Incidence berbasis brand_combo hasil auto-labeling RM1. Exclusive count selalu berjumlah total HCC valid; multi-label incidence dapat melebihi total HCC karena HCC mixed dihitung sekali pada tiap brand aktif.

Pengelompokan brand pada HCC menunjukkan konteks video yang dikomentari oleh anggota komunitas berdasarkan hashtag metadata video. Label tersebut tidak berarti seluruh anggota HCC menulis hashtag, mendukung brand, berafiliasi dengan brand, atau membahas brand secara eksplisit dalam setiap komentar.

## 22. Video-Normalized Analysis

Notebook menghitung comments_per_active_video dan community_accounts_per_active_video untuk seluruh komentar, Community Actor, dan setiap konteks brand per weekday.

## 23. Individual Actor Temporal Profile

Komentar kreator dan waktu upload video dipisahkan. Waktu upload video tidak disamakan dengan waktu komentar dan tidak dipakai untuk menyimpulkan jadwal atau goals kreator.

## 24. HCC Temporal-Goals Integration

Profil temporal HCC digabung dengan goal_orientation, dominant sentiment, rasio sentimen, primary_brand, brand_label_auto, dan narrative_similarity_level, disertai interpretation_safe.

## 25. Statistical Comparison

Uji chi-square actor_type_primary x weekday_name_id disertai Cramer's V dan standardized residual. P-value tidak digunakan sebagai satu-satunya dasar interpretasi.

## 26. Visualizations

Semua visualisasi disimpan sebagai PNG di output/rm2_temporal/visualisasi/. Judul visualisasi menyebut unit penghitungan: jumlah komentar, jumlah HCC, jumlah akun, atau lift.

## 27. Gephi Export

Node HCC temporal diekspor dengan atribut temporal baru. Edge HCC disalin tanpa perubahan dan hash edge output divalidasi sama dengan hash edge sumber.

## 28. Interpretation Guide

Dominant weekday, peak hour, lift, entropy, dan recurrence adalah pola temporal yang teramati. Hasil tersebut harus dibaca bersama denominator, active weeks, reliability, account-level profile, dan konteks brand.

## 29. Limitations

Community Actor merupakan akun anggota HCC, bukan akun yang telah terbukti sebagai buzzer. Aktivitas yang terkonsentrasi pada hari atau jam tertentu tidak membuktikan jadwal kerja, pembayaran, hubungan komersial, afiliasi brand, atau koordinasi terencana.

## 30. Final Validation Report

Cell berikut menjalankan seluruh pipeline dan mencetak ringkasan final: timestamp coverage, rentang tanggal, sensitivity WIB-WITA, distribusi komentar, weekday lift, HCC profiles, brand-HCC summaries, visualisasi, Gephi export, dan konfirmasi checksum input.

In [2]:
results = run_pipeline()

FINAL VALIDATION REPORT
Valid timestamp coverage: 99.83%
UTC range: 2022-07-21 12:34:13+00:00 to 2026-06-12 11:45:49+00:00
WIB range: 2022-07-21 19:34:13+07:00 to 2026-06-12 18:45:49+07:00
WIB-WITA changed weekday percentage: 3.07%

Comments per actor type:
actor_type_primary
Individual Actor     1384
Community Actor      1009
Mass Actor          31454

Dominant weekday per actor type:
actor_type_primary weekday_name_id  n_comments
   Community Actor           Kamis         177
  Individual Actor          Selasa         222
        Mass Actor          Selasa        6220

Community Actor weekday lift:
weekday  community_comment_count  weekday_activity_lift
  Senin                      128               1.155161
 Selasa                      162               0.872463
   Rabu                      142               0.992809
  Kamis                      177               1.046938
  Jumat                      109               0.924351
  Sabtu                      145               1.229935
